## Prompt 1

Agisci come uno sviluppatore Python. Scrivimi un'app Python che gestisca la modifica di una foto. Deve avere un'interfaccia utente intuitiva e moderna utilizzando la libreria tkinter. L'utente deve poter caricare l'immagine che sarà visualizzata a schermo. Sotto di essa ci saranno i vari pulsanti per le modifiche, che l'utente potrà controllare in tempo reale:      
- filtro vintage: applica un effetto seppia all'immagine       
- filtro b/n: applica il classico effetto bianco e nero       
- ritaglio dell'immagine      
- luminosità (con slider)      
- contrasto (con slider)       
- temperatura (con slider)      
- specchio       
- rotazione       

Vorrei infine che ci sia la possibilità di salvare l'immagine sia in formato jpg che in png.

In [1]:
"""
Editor di Foto Professionale
Applicazione moderna per la modifica di immagini con interfaccia intuitiva
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk, ImageEnhance, ImageFilter, ImageOps
import numpy as np
from typing import Optional
import os


class PhotoEditor:
    """Editor di foto con interfaccia grafica moderna"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Editor di Foto Professionale")
        self.root.geometry("1200x800")
        self.root.configure(bg="#1a1a1a")
        
        # Variabili per le immagini
        self.original_image: Optional[Image.Image] = None
        self.current_image: Optional[Image.Image] = None
        self.display_image: Optional[ImageTk.PhotoImage] = None
        self.crop_mode = False
        self.crop_start = None
        self.crop_rect = None
        
        # Variabili per i filtri
        self.brightness_var = tk.DoubleVar(value=1.0)
        self.contrast_var = tk.DoubleVar(value=1.0)
        self.temperature_var = tk.DoubleVar(value=0)
        
        # Configurazione dello stile moderno
        self.setup_styles()
        
        # Creazione dell'interfaccia
        self.create_widgets()
        
        # Binding degli eventi
        self.brightness_var.trace_add('write', self.apply_adjustments)
        self.contrast_var.trace_add('write', self.apply_adjustments)
        self.temperature_var.trace_add('write', self.apply_adjustments)
        
    def setup_styles(self):
        """Configura gli stili moderni per l'interfaccia"""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colori moderni
        bg_dark = "#1a1a1a"
        bg_medium = "#2d2d2d"
        bg_light = "#3d3d3d"
        accent = "#4a9eff"
        text_color = "#ffffff"
        
        # Stile per i pulsanti
        style.configure('Modern.TButton',
                       background=bg_medium,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10))
        style.map('Modern.TButton',
                 background=[('active', bg_light), ('pressed', accent)])
        
        # Stile per le etichette
        style.configure('Modern.TLabel',
                       background=bg_dark,
                       foreground=text_color,
                       font=('Segoe UI', 10))
        
        # Stile per i frame
        style.configure('Modern.TFrame',
                       background=bg_dark)
        
        # Stile per gli slider
        style.configure('Modern.Horizontal.TScale',
                       background=bg_dark,
                       troughcolor=bg_medium,
                       borderwidth=0,
                       sliderthickness=20)
        
    def create_widgets(self):
        """Crea tutti i widget dell'interfaccia"""
        # Frame principale
        main_frame = ttk.Frame(self.root, style='Modern.TFrame')
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Area superiore: canvas per l'immagine
        self.canvas_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        self.canvas_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 10))
        
        self.canvas = tk.Canvas(self.canvas_frame,
                               bg="#2d2d2d",
                               highlightthickness=0,
                               cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Testo placeholder
        self.canvas.create_text(
            400, 300,
            text="Carica un'immagine per iniziare",
            fill="#666666",
            font=('Segoe UI', 16),
            tags="placeholder"
        )
        
        # Binding per il crop
        self.canvas.bind("<Button-1>", self.on_crop_start)
        self.canvas.bind("<B1-Motion>", self.on_crop_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_end)
        
        # Area inferiore: controlli
        controls_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        controls_frame.pack(fill=tk.X)
        
        # Frame per i pulsanti principali
        buttons_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        buttons_frame.pack(fill=tk.X, pady=(0, 10))
        
        # Pulsante carica
        load_btn = ttk.Button(buttons_frame,
                             text="📁 Carica Immagine",
                             command=self.load_image,
                             style='Modern.TButton')
        load_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante salva JPG
        save_jpg_btn = ttk.Button(buttons_frame,
                                 text="💾 Salva JPG",
                                 command=lambda: self.save_image('jpg'),
                                 style='Modern.TButton')
        save_jpg_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante salva PNG
        save_png_btn = ttk.Button(buttons_frame,
                                 text="💾 Salva PNG",
                                 command=lambda: self.save_image('png'),
                                 style='Modern.TButton')
        save_png_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsanti filtri
        vintage_btn = ttk.Button(buttons_frame,
                                text="🎨 Vintage",
                                command=self.apply_vintage,
                                style='Modern.TButton')
        vintage_btn.pack(side=tk.LEFT, padx=5)
        
        bw_btn = ttk.Button(buttons_frame,
                           text="⚫ Bianco e Nero",
                           command=self.apply_bw,
                           style='Modern.TButton')
        bw_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsanti trasformazione
        crop_btn = ttk.Button(buttons_frame,
                             text="✂️ Ritaglia",
                             command=self.toggle_crop_mode,
                             style='Modern.TButton')
        crop_btn.pack(side=tk.LEFT, padx=5)
        
        mirror_btn = ttk.Button(buttons_frame,
                               text="🔄 Specchio",
                               command=self.apply_mirror,
                               style='Modern.TButton')
        mirror_btn.pack(side=tk.LEFT, padx=5)
        
        rotate_btn = ttk.Button(buttons_frame,
                               text="🔃 Ruota 90°",
                               command=self.apply_rotation,
                               style='Modern.TButton')
        rotate_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsante reset
        reset_btn = ttk.Button(buttons_frame,
                              text="↺ Reset",
                              command=self.reset_image,
                              style='Modern.TButton')
        reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Frame per gli slider
        sliders_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        sliders_frame.pack(fill=tk.X)
        
        # Slider luminosità
        self.create_slider(sliders_frame, "☀️ Luminosità", self.brightness_var, 0.0, 2.0, 0)
        
        # Slider contrasto
        self.create_slider(sliders_frame, "◐ Contrasto", self.contrast_var, 0.0, 2.0, 1)
        
        # Slider temperatura
        self.create_slider(sliders_frame, "🌡️ Temperatura", self.temperature_var, -100, 100, 2)
        
    def create_slider(self, parent, label_text, variable, from_, to, column):
        """Crea uno slider con etichetta"""
        frame = ttk.Frame(parent, style='Modern.TFrame')
        frame.grid(row=0, column=column, padx=10, pady=5, sticky='ew')
        parent.columnconfigure(column, weight=1)
        
        label = ttk.Label(frame, text=label_text, style='Modern.TLabel')
        label.pack(anchor='w')
        
        slider = ttk.Scale(frame,
                          from_=from_,
                          to=to,
                          variable=variable,
                          orient=tk.HORIZONTAL,
                          style='Modern.Horizontal.TScale')
        slider.pack(fill=tk.X, pady=5)
        
        # Etichetta valore
        value_label = ttk.Label(frame, text=f"{variable.get():.2f}", style='Modern.TLabel')
        value_label.pack(anchor='e')
        
        def update_label(*args):
            value_label.config(text=f"{variable.get():.2f}")
        
        variable.trace_add('write', update_label)
        
    def load_image(self):
        """Carica un'immagine dal file system"""
        file_path = filedialog.askopenfilename(
            title="Seleziona un'immagine",
            filetypes=[
                ("Immagini", "*.jpg *.jpeg *.png *.bmp *.gif"),
                ("JPEG", "*.jpg *.jpeg"),
                ("PNG", "*.png"),
                ("Tutti i file", "*.*")
            ]
        )
        
        if file_path:
            try:
                self.original_image = Image.open(file_path)
                self.current_image = self.original_image.copy()
                
                # Reset dei controlli
                self.brightness_var.set(1.0)
                self.contrast_var.set(1.0)
                self.temperature_var.set(0)
                
                self.display_current_image()
                self.canvas.delete("placeholder")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile caricare l'immagine:\n{str(e)}")
                
    def display_current_image(self):
        """Visualizza l'immagine corrente sul canvas"""
        if self.current_image is None:
            return
            
        # Calcola le dimensioni per il fit
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
            
        # Calcola il ridimensionamento mantenendo l'aspect ratio
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        new_width = int(img_width * ratio * 0.9)  # 90% per margini
        new_height = int(img_height * ratio * 0.9)
        
        # Ridimensiona l'immagine
        display_img = self.current_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Converti in PhotoImage
        self.display_image = ImageTk.PhotoImage(display_img)
        
        # Pulisci il canvas e mostra l'immagine
        self.canvas.delete("all")
        self.canvas.create_image(
            canvas_width // 2,
            canvas_height // 2,
            image=self.display_image,
            anchor=tk.CENTER,
            tags="image"
        )
        
    def apply_adjustments(self, *args):
        """Applica le regolazioni di luminosità, contrasto e temperatura"""
        if self.original_image is None:
            return
            
        # Parte dall'immagine originale
        img = self.original_image.copy()
        
        # Applica luminosità
        if self.brightness_var.get() != 1.0:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(self.brightness_var.get())
            
        # Applica contrasto
        if self.contrast_var.get() != 1.0:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(self.contrast_var.get())
            
        # Applica temperatura (modifica del bilanciamento del colore)
        if self.temperature_var.get() != 0:
            img = self.adjust_temperature(img, self.temperature_var.get())
            
        self.current_image = img
        self.display_current_image()
        
    def adjust_temperature(self, image, value):
        """Regola la temperatura del colore dell'immagine"""
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        # Converti in array numpy
        img_array = np.array(image, dtype=np.float32)
        
        # Applica shift di temperatura
        if value > 0:  # Più caldo (più rosso/giallo)
            img_array[:, :, 0] += value * 0.5  # Rosso
            img_array[:, :, 1] += value * 0.3  # Verde
        else:  # Più freddo (più blu)
            img_array[:, :, 2] += abs(value) * 0.5  # Blu
            
        # Clamp values
        img_array = np.clip(img_array, 0, 255)
        
        return Image.fromarray(img_array.astype(np.uint8))
        
    def apply_vintage(self):
        """Applica un filtro vintage (seppia)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        img = self.current_image.convert('RGB')
        img_array = np.array(img, dtype=np.float32)
        
        # Matrice seppia
        sepia_filter = np.array([
            [0.393, 0.769, 0.189],
            [0.349, 0.686, 0.168],
            [0.272, 0.534, 0.131]
        ])
        
        # Applica il filtro
        sepia_img = img_array @ sepia_filter.T
        sepia_img = np.clip(sepia_img, 0, 255)
        
        self.original_image = Image.fromarray(sepia_img.astype(np.uint8))
        self.apply_adjustments()
        
    def apply_bw(self):
        """Applica il filtro bianco e nero"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.original_image = self.current_image.convert('L').convert('RGB')
        self.apply_adjustments()
        
    def toggle_crop_mode(self):
        """Attiva/disattiva la modalità ritaglio"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.crop_mode = not self.crop_mode
        if self.crop_mode:
            self.canvas.config(cursor="cross")
            messagebox.showinfo("Modalità Ritaglio",
                              "Trascina il mouse sull'immagine per selezionare l'area da ritagliare.\n"
                              "Clicca di nuovo su 'Ritaglia' per applicare.")
        else:
            self.canvas.config(cursor="arrow")
            if self.crop_rect:
                self.apply_crop()
                
    def on_crop_start(self, event):
        """Inizia la selezione per il ritaglio"""
        if not self.crop_mode:
            return
        self.crop_start = (event.x, event.y)
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
    def on_crop_drag(self, event):
        """Disegna il rettangolo di selezione"""
        if not self.crop_mode or not self.crop_start:
            return
            
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
        self.crop_rect = self.canvas.create_rectangle(
            self.crop_start[0], self.crop_start[1],
            event.x, event.y,
            outline='#4a9eff',
            width=2,
            dash=(5, 5)
        )
        
    def on_crop_end(self, event):
        """Finalizza la selezione"""
        if not self.crop_mode or not self.crop_start:
            return
        self.crop_end = (event.x, event.y)
        
    def apply_crop(self):
        """Applica il ritaglio all'immagine"""
        if not self.crop_start or not self.crop_end:
            return
            
        # Calcola le coordinate relative all'immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Converti coordinate canvas in coordinate immagine
        x1 = int((min(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y1 = int((min(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        x2 = int((max(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y2 = int((max(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        
        # Clamp ai limiti dell'immagine
        x1 = max(0, min(x1, img_width))
        y1 = max(0, min(y1, img_height))
        x2 = max(0, min(x2, img_width))
        y2 = max(0, min(y2, img_height))
        
        if x2 > x1 and y2 > y1:
            self.original_image = self.current_image.crop((x1, y1, x2, y2))
            self.apply_adjustments()
            
        self.crop_mode = False
        self.crop_start = None
        self.crop_end = None
        self.crop_rect = None
        self.canvas.config(cursor="arrow")
        
    def apply_mirror(self):
        """Applica l'effetto specchio (flip orizzontale)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.original_image = self.current_image.transpose(Image.FLIP_LEFT_RIGHT)
        self.apply_adjustments()
        
    def apply_rotation(self):
        """Ruota l'immagine di 90 gradi in senso orario"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.original_image = self.current_image.rotate(-90, expand=True)
        self.apply_adjustments()
        
    def reset_image(self):
        """Resetta tutte le modifiche"""
        if self.original_image is None:
            return
            
        self.brightness_var.set(1.0)
        self.contrast_var.set(1.0)
        self.temperature_var.set(0)
        self.apply_adjustments()
        
    def save_image(self, format_type):
        """Salva l'immagine nel formato specificato"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Nessuna immagine da salvare!")
            return
            
        file_extension = f".{format_type}"
        file_types = [(format_type.upper(), f"*{file_extension}")]
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=file_extension,
            filetypes=file_types,
            title=f"Salva come {format_type.upper()}"
        )
        
        if file_path:
            try:
                if format_type.lower() == 'jpg':
                    # Converti in RGB se necessario (JPG non supporta trasparenza)
                    img_to_save = self.current_image
                    if img_to_save.mode in ('RGBA', 'LA', 'P'):
                        rgb_img = Image.new('RGB', img_to_save.size, (255, 255, 255))
                        if img_to_save.mode == 'P':
                            img_to_save = img_to_save.convert('RGBA')
                        rgb_img.paste(img_to_save, mask=img_to_save.split()[-1] if img_to_save.mode == 'RGBA' else None)
                        img_to_save = rgb_img
                    img_to_save.save(file_path, 'JPEG', quality=95)
                else:
                    self.current_image.save(file_path, 'PNG')
                    
                messagebox.showinfo("Successo", f"Immagine salvata con successo in:\n{file_path}")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile salvare l'immagine:\n{str(e)}")


def main():
    """Funzione principale"""
    root = tk.Tk()
    app = PhotoEditor(root)
    
    # Configura il ridimensionamento della finestra
    root.minsize(800, 600)
    
    # Centra la finestra
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()


## Prompt 2

Va quasi tutto bene ma dobbiamo modificare i seguenti aspetti:       
- Per i pulsanti vintage e b/n il filtro deve poter essere applicato una volta sola, e se rischiaccio il bottone il filtro viene rimosso.     
- Fare in modo di indicare il bottone premuto o no, cambiando ad esempio lo sfondo del bottone.      
- Il bottone reset deve resettare TUTTI i filtri, non solo quelli dei 3 slider, e ritornare all'immagine originale.      
- Aggiungere il bottone Undo che annulla l'ultima modifica fatta.     

In [2]:
"""
Editor di Foto Professionale
Applicazione moderna per la modifica di immagini con interfaccia intuitiva
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk, ImageEnhance, ImageFilter, ImageOps
import numpy as np
from typing import Optional
import os


class PhotoEditor:
    """Editor di foto con interfaccia grafica moderna"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Editor di Foto Professionale")
        self.root.geometry("1200x800")
        self.root.configure(bg="#1a1a1a")
        
        # Variabili per le immagini
        self.original_image: Optional[Image.Image] = None
        self.current_image: Optional[Image.Image] = None
        self.display_image: Optional[ImageTk.PhotoImage] = None
        self.crop_mode = False
        self.crop_start = None
        self.crop_rect = None
        
        # Cronologia per Undo (stack di immagini)
        self.history = []
        self.max_history = 20  # Massimo 20 stati nella cronologia
        
        # Variabili per i filtri
        self.brightness_var = tk.DoubleVar(value=1.0)
        self.contrast_var = tk.DoubleVar(value=1.0)
        self.temperature_var = tk.DoubleVar(value=0)
        
        # Stati dei filtri toggle
        self.vintage_active = False
        self.bw_active = False
        
        # Riferimenti ai pulsanti per cambio colore
        self.vintage_btn = None
        self.bw_btn = None
        
        # Configurazione dello stile moderno
        self.setup_styles()
        
        # Creazione dell'interfaccia
        self.create_widgets()
        
        # Binding degli eventi
        self.brightness_var.trace_add('write', self.apply_adjustments)
        self.contrast_var.trace_add('write', self.apply_adjustments)
        self.temperature_var.trace_add('write', self.apply_adjustments)
        
    def setup_styles(self):
        """Configura gli stili moderni per l'interfaccia"""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colori moderni
        bg_dark = "#1a1a1a"
        bg_medium = "#2d2d2d"
        bg_light = "#3d3d3d"
        accent = "#4a9eff"
        text_color = "#ffffff"
        
        # Stile per i pulsanti
        style.configure('Modern.TButton',
                       background=bg_medium,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10))
        style.map('Modern.TButton',
                 background=[('active', bg_light), ('pressed', accent)])
        
        # Stile per i pulsanti attivi (filtri applicati)
        style.configure('Active.TButton',
                       background=accent,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10, 'bold'))
        style.map('Active.TButton',
                 background=[('active', '#3a8eef'), ('pressed', '#2a7edf')])
        
        # Stile per le etichette
        style.configure('Modern.TLabel',
                       background=bg_dark,
                       foreground=text_color,
                       font=('Segoe UI', 10))
        
        # Stile per i frame
        style.configure('Modern.TFrame',
                       background=bg_dark)
        
        # Stile per gli slider
        style.configure('Modern.Horizontal.TScale',
                       background=bg_dark,
                       troughcolor=bg_medium,
                       borderwidth=0,
                       sliderthickness=20)
        
    def create_widgets(self):
        """Crea tutti i widget dell'interfaccia"""
        # Frame principale
        main_frame = ttk.Frame(self.root, style='Modern.TFrame')
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Area superiore: canvas per l'immagine
        self.canvas_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        self.canvas_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 10))
        
        self.canvas = tk.Canvas(self.canvas_frame,
                               bg="#2d2d2d",
                               highlightthickness=0,
                               cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Testo placeholder
        self.canvas.create_text(
            400, 300,
            text="Carica un'immagine per iniziare",
            fill="#666666",
            font=('Segoe UI', 16),
            tags="placeholder"
        )
        
        # Binding per il crop
        self.canvas.bind("<Button-1>", self.on_crop_start)
        self.canvas.bind("<B1-Motion>", self.on_crop_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_end)
        
        # Area inferiore: controlli
        controls_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        controls_frame.pack(fill=tk.X)
        
        # Frame per i pulsanti principali
        buttons_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        buttons_frame.pack(fill=tk.X, pady=(0, 10))
        
        # Pulsante carica
        load_btn = ttk.Button(buttons_frame,
                             text="📁 Carica Immagine",
                             command=self.load_image,
                             style='Modern.TButton')
        load_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante salva JPG
        save_jpg_btn = ttk.Button(buttons_frame,
                                 text="💾 Salva JPG",
                                 command=lambda: self.save_image('jpg'),
                                 style='Modern.TButton')
        save_jpg_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante salva PNG
        save_png_btn = ttk.Button(buttons_frame,
                                 text="💾 Salva PNG",
                                 command=lambda: self.save_image('png'),
                                 style='Modern.TButton')
        save_png_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsanti filtri
        self.vintage_btn = ttk.Button(buttons_frame,
                                text="🎨 Vintage",
                                command=self.toggle_vintage,
                                style='Modern.TButton')
        self.vintage_btn.pack(side=tk.LEFT, padx=5)
        
        self.bw_btn = ttk.Button(buttons_frame,
                           text="⚫ Bianco e Nero",
                           command=self.toggle_bw,
                           style='Modern.TButton')
        self.bw_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsanti trasformazione
        crop_btn = ttk.Button(buttons_frame,
                             text="✂️ Ritaglia",
                             command=self.toggle_crop_mode,
                             style='Modern.TButton')
        crop_btn.pack(side=tk.LEFT, padx=5)
        
        mirror_btn = ttk.Button(buttons_frame,
                               text="🔄 Specchio",
                               command=self.apply_mirror,
                               style='Modern.TButton')
        mirror_btn.pack(side=tk.LEFT, padx=5)
        
        rotate_btn = ttk.Button(buttons_frame,
                               text="🔃 Ruota 90°",
                               command=self.apply_rotation,
                               style='Modern.TButton')
        rotate_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsante Undo
        undo_btn = ttk.Button(buttons_frame,
                             text="⮌ Undo",
                             command=self.undo,
                             style='Modern.TButton')
        undo_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante reset
        reset_btn = ttk.Button(buttons_frame,
                              text="↺ Reset",
                              command=self.reset_image,
                              style='Modern.TButton')
        reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Frame per gli slider
        sliders_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        sliders_frame.pack(fill=tk.X)
        
        # Slider luminosità
        self.create_slider(sliders_frame, "☀️ Luminosità", self.brightness_var, 0.0, 2.0, 0)
        
        # Slider contrasto
        self.create_slider(sliders_frame, "◐ Contrasto", self.contrast_var, 0.0, 2.0, 1)
        
        # Slider temperatura
        self.create_slider(sliders_frame, "🌡️ Temperatura", self.temperature_var, -100, 100, 2)
        
    def create_slider(self, parent, label_text, variable, from_, to, column):
        """Crea uno slider con etichetta"""
        frame = ttk.Frame(parent, style='Modern.TFrame')
        frame.grid(row=0, column=column, padx=10, pady=5, sticky='ew')
        parent.columnconfigure(column, weight=1)
        
        label = ttk.Label(frame, text=label_text, style='Modern.TLabel')
        label.pack(anchor='w')
        
        slider = ttk.Scale(frame,
                          from_=from_,
                          to=to,
                          variable=variable,
                          orient=tk.HORIZONTAL,
                          style='Modern.Horizontal.TScale')
        slider.pack(fill=tk.X, pady=5)
        
        # Etichetta valore
        value_label = ttk.Label(frame, text=f"{variable.get():.2f}", style='Modern.TLabel')
        value_label.pack(anchor='e')
        
        def update_label(*args):
            value_label.config(text=f"{variable.get():.2f}")
        
        variable.trace_add('write', update_label)
        
    def load_image(self):
        """Carica un'immagine dal file system"""
        file_path = filedialog.askopenfilename(
            title="Seleziona un'immagine",
            filetypes=[
                ("Immagini", "*.jpg *.jpeg *.png *.bmp *.gif"),
                ("JPEG", "*.jpg *.jpeg"),
                ("PNG", "*.png"),
                ("Tutti i file", "*.*")
            ]
        )
        
        if file_path:
            try:
                self.original_image = Image.open(file_path)
                self.current_image = self.original_image.copy()
                
                # Reset di tutti i controlli e filtri
                self.brightness_var.set(1.0)
                self.contrast_var.set(1.0)
                self.temperature_var.set(0)
                
                # Reset stato filtri
                self.vintage_active = False
                self.bw_active = False
                self.update_filter_buttons()
                
                # Reset cronologia
                self.history = []
                self.save_to_history()
                
                self.display_current_image()
                self.canvas.delete("placeholder")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile caricare l'immagine:\n{str(e)}")
                
    def display_current_image(self):
        """Visualizza l'immagine corrente sul canvas"""
        if self.current_image is None:
            return
            
        # Calcola le dimensioni per il fit
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
            
        # Calcola il ridimensionamento mantenendo l'aspect ratio
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        new_width = int(img_width * ratio * 0.9)  # 90% per margini
        new_height = int(img_height * ratio * 0.9)
        
        # Ridimensiona l'immagine
        display_img = self.current_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Converti in PhotoImage
        self.display_image = ImageTk.PhotoImage(display_img)
        
        # Pulisci il canvas e mostra l'immagine
        self.canvas.delete("all")
        self.canvas.create_image(
            canvas_width // 2,
            canvas_height // 2,
            image=self.display_image,
            anchor=tk.CENTER,
            tags="image"
        )
        
    def save_to_history(self):
        """Salva lo stato corrente nella cronologia"""
        if self.original_image is None:
            return
            
        # Salva una copia dell'immagine e dello stato dei filtri
        state = {
            'image': self.original_image.copy(),
            'brightness': self.brightness_var.get(),
            'contrast': self.contrast_var.get(),
            'temperature': self.temperature_var.get(),
            'vintage': self.vintage_active,
            'bw': self.bw_active
        }
        
        self.history.append(state)
        
        # Limita la dimensione della cronologia
        if len(self.history) > self.max_history:
            self.history.pop(0)
            
    def undo(self):
        """Annulla l'ultima modifica"""
        if len(self.history) <= 1:
            messagebox.showinfo("Undo", "Nessuna operazione da annullare!")
            return
            
        # Rimuovi lo stato corrente
        self.history.pop()
        
        # Ripristina lo stato precedente
        if self.history:
            previous_state = self.history[-1]
            self.original_image = previous_state['image'].copy()
            self.brightness_var.set(previous_state['brightness'])
            self.contrast_var.set(previous_state['contrast'])
            self.temperature_var.set(previous_state['temperature'])
            self.vintage_active = previous_state['vintage']
            self.bw_active = previous_state['bw']
            
            self.update_filter_buttons()
            self.apply_adjustments()
            
    def update_filter_buttons(self):
        """Aggiorna lo stile dei pulsanti filtro in base allo stato"""
        if self.vintage_btn:
            style = 'Active.TButton' if self.vintage_active else 'Modern.TButton'
            self.vintage_btn.configure(style=style)
            
        if self.bw_btn:
            style = 'Active.TButton' if self.bw_active else 'Modern.TButton'
            self.bw_btn.configure(style=style)
        
    def apply_adjustments(self, *args):
        """Applica le regolazioni di luminosità, contrasto e temperatura"""
        if self.original_image is None:
            return
            
        # Parte dall'immagine originale
        img = self.original_image.copy()
        
        # Applica luminosità
        if self.brightness_var.get() != 1.0:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(self.brightness_var.get())
            
        # Applica contrasto
        if self.contrast_var.get() != 1.0:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(self.contrast_var.get())
            
        # Applica temperatura (modifica del bilanciamento del colore)
        if self.temperature_var.get() != 0:
            img = self.adjust_temperature(img, self.temperature_var.get())
            
        self.current_image = img
        self.display_current_image()
        
    def adjust_temperature(self, image, value):
        """Regola la temperatura del colore dell'immagine"""
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        # Converti in array numpy
        img_array = np.array(image, dtype=np.float32)
        
        # Applica shift di temperatura
        if value > 0:  # Più caldo (più rosso/giallo)
            img_array[:, :, 0] += value * 0.5  # Rosso
            img_array[:, :, 1] += value * 0.3  # Verde
        else:  # Più freddo (più blu)
            img_array[:, :, 2] += abs(value) * 0.5  # Blu
            
        # Clamp values
        img_array = np.clip(img_array, 0, 255)
        
        return Image.fromarray(img_array.astype(np.uint8))
        
    def toggle_vintage(self):
        """Attiva/disattiva il filtro vintage (seppia)"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Salva lo stato corrente prima di modificare
        self.save_to_history()
        
        # Toggle dello stato
        self.vintage_active = not self.vintage_active
        
        if self.vintage_active:
            # Disattiva B/N se attivo
            if self.bw_active:
                self.bw_active = False
            
            # Applica filtro vintage
            img = self.original_image.convert('RGB')
            img_array = np.array(img, dtype=np.float32)
            
            # Matrice seppia
            sepia_filter = np.array([
                [0.393, 0.769, 0.189],
                [0.349, 0.686, 0.168],
                [0.272, 0.534, 0.131]
            ])
            
            # Applica il filtro
            sepia_img = img_array @ sepia_filter.T
            sepia_img = np.clip(sepia_img, 0, 255)
            
            self.original_image = Image.fromarray(sepia_img.astype(np.uint8))
        else:
            # Rimuovi il filtro tornando all'immagine dalla cronologia
            if len(self.history) >= 2:
                # Trova l'ultimo stato senza vintage
                for i in range(len(self.history) - 2, -1, -1):
                    if not self.history[i]['vintage']:
                        self.original_image = self.history[i]['image'].copy()
                        break
        
        self.update_filter_buttons()
        self.apply_adjustments()
        
    def toggle_bw(self):
        """Attiva/disattiva il filtro bianco e nero"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Salva lo stato corrente prima di modificare
        self.save_to_history()
        
        # Toggle dello stato
        self.bw_active = not self.bw_active
        
        if self.bw_active:
            # Disattiva Vintage se attivo
            if self.vintage_active:
                self.vintage_active = False
            
            # Applica filtro bianco e nero
            self.original_image = self.original_image.convert('L').convert('RGB')
        else:
            # Rimuovi il filtro tornando all'immagine dalla cronologia
            if len(self.history) >= 2:
                # Trova l'ultimo stato senza b/n
                for i in range(len(self.history) - 2, -1, -1):
                    if not self.history[i]['bw']:
                        self.original_image = self.history[i]['image'].copy()
                        break
        
        self.update_filter_buttons()
        self.apply_adjustments()
        
    def toggle_crop_mode(self):
        """Attiva/disattiva la modalità ritaglio"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.crop_mode = not self.crop_mode
        if self.crop_mode:
            self.canvas.config(cursor="cross")
            messagebox.showinfo("Modalità Ritaglio",
                              "Trascina il mouse sull'immagine per selezionare l'area da ritagliare.\n"
                              "Clicca di nuovo su 'Ritaglia' per applicare.")
        else:
            self.canvas.config(cursor="arrow")
            if self.crop_rect:
                self.apply_crop()
                
    def on_crop_start(self, event):
        """Inizia la selezione per il ritaglio"""
        if not self.crop_mode:
            return
        self.crop_start = (event.x, event.y)
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
    def on_crop_drag(self, event):
        """Disegna il rettangolo di selezione"""
        if not self.crop_mode or not self.crop_start:
            return
            
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
        self.crop_rect = self.canvas.create_rectangle(
            self.crop_start[0], self.crop_start[1],
            event.x, event.y,
            outline='#4a9eff',
            width=2,
            dash=(5, 5)
        )
        
    def on_crop_end(self, event):
        """Finalizza la selezione"""
        if not self.crop_mode or not self.crop_start:
            return
        self.crop_end = (event.x, event.y)
        
    def apply_crop(self):
        """Applica il ritaglio all'immagine"""
        if not self.crop_start or not self.crop_end:
            return
            
        # Calcola le coordinate relative all'immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Converti coordinate canvas in coordinate immagine
        x1 = int((min(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y1 = int((min(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        x2 = int((max(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y2 = int((max(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        
        # Clamp ai limiti dell'immagine
        x1 = max(0, min(x1, img_width))
        y1 = max(0, min(y1, img_height))
        x2 = max(0, min(x2, img_width))
        y2 = max(0, min(y2, img_height))
        
        if x2 > x1 and y2 > y1:
            self.save_to_history()
            self.original_image = self.current_image.crop((x1, y1, x2, y2))
            self.apply_adjustments()
            
        self.crop_mode = False
        self.crop_start = None
        self.crop_end = None
        self.crop_rect = None
        self.canvas.config(cursor="arrow")
        
    def apply_mirror(self):
        """Applica l'effetto specchio (flip orizzontale)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.save_to_history()
        self.original_image = self.current_image.transpose(Image.FLIP_LEFT_RIGHT)
        self.apply_adjustments()
        
    def apply_rotation(self):
        """Ruota l'immagine di 90 gradi in senso orario"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.save_to_history()
        self.original_image = self.current_image.rotate(-90, expand=True)
        self.apply_adjustments()
        
    def reset_image(self):
        """Resetta tutte le modifiche e torna all'immagine originale"""
        if not self.history:
            return
        
        # Torna alla primissima immagine caricata
        first_state = self.history[0]
        self.original_image = first_state['image'].copy()
        
        # Reset di tutti i controlli
        self.brightness_var.set(1.0)
        self.contrast_var.set(1.0)
        self.temperature_var.set(0)
        
        # Reset di tutti i filtri
        self.vintage_active = False
        self.bw_active = False
        self.update_filter_buttons()
        
        # Resetta la cronologia mantenendo solo lo stato originale
        self.history = [first_state]
        
        self.apply_adjustments()
        
    def save_image(self, format_type):
        """Salva l'immagine nel formato specificato"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Nessuna immagine da salvare!")
            return
            
        file_extension = f".{format_type}"
        file_types = [(format_type.upper(), f"*{file_extension}")]
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=file_extension,
            filetypes=file_types,
            title=f"Salva come {format_type.upper()}"
        )
        
        if file_path:
            try:
                if format_type.lower() == 'jpg':
                    # Converti in RGB se necessario (JPG non supporta trasparenza)
                    img_to_save = self.current_image
                    if img_to_save.mode in ('RGBA', 'LA', 'P'):
                        rgb_img = Image.new('RGB', img_to_save.size, (255, 255, 255))
                        if img_to_save.mode == 'P':
                            img_to_save = img_to_save.convert('RGBA')
                        rgb_img.paste(img_to_save, mask=img_to_save.split()[-1] if img_to_save.mode == 'RGBA' else None)
                        img_to_save = rgb_img
                    img_to_save.save(file_path, 'JPEG', quality=95)
                else:
                    self.current_image.save(file_path, 'PNG')
                    
                messagebox.showinfo("Successo", f"Immagine salvata con successo in:\n{file_path}")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile salvare l'immagine:\n{str(e)}")


def main():
    """Funzione principale"""
    root = tk.Tk()
    app = PhotoEditor(root)
    
    # Configura il ridimensionamento della finestra
    root.minsize(800, 600)
    
    # Centra la finestra
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()


## Prompt 3

Ci sono i seguenti problemi che andrebbero risolti:       
- L’Undo si perde sempre una operazione se lo clicco n volte (con n operazioni) per annullarle tutte a ritroso.      
- I filtri vintage e b/n devono essere mutuamente esclusivi (adesso invece quando clicco vintage e poi clicco bianco e nero è come se me li applicasse uno sopra l’altro). Quindi vorrei che quando è selezionato vintage, se selezionassi b/n prima andrebbe tolto in automatico vintage.         
- I bottoni se la finestra dell’app non è estesa per tutta la larghezza dello schermo non sono visibili. Vorrei che l’interfaccia sia responsive.            

In [3]:
"""
Editor di Foto Professionale
Applicazione moderna per la modifica di immagini con interfaccia intuitiva
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk, ImageEnhance, ImageFilter, ImageOps
import numpy as np
from typing import Optional
import os


class PhotoEditor:
    """Editor di foto con interfaccia grafica moderna"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Editor di Foto Professionale")
        self.root.geometry("1200x800")
        self.root.configure(bg="#1a1a1a")
        
        # Variabili per le immagini
        self.original_image: Optional[Image.Image] = None
        self.current_image: Optional[Image.Image] = None
        self.display_image: Optional[ImageTk.PhotoImage] = None
        self.crop_mode = False
        self.crop_start = None
        self.crop_rect = None
        
        # Cronologia per Undo (stack di immagini)
        self.history = []
        self.max_history = 20  # Massimo 20 stati nella cronologia
        
        # Variabili per i filtri
        self.brightness_var = tk.DoubleVar(value=1.0)
        self.contrast_var = tk.DoubleVar(value=1.0)
        self.temperature_var = tk.DoubleVar(value=0)
        
        # Stati dei filtri toggle
        self.vintage_active = False
        self.bw_active = False
        
        # Riferimenti ai pulsanti per cambio colore
        self.vintage_btn = None
        self.bw_btn = None
        
        # Configurazione dello stile moderno
        self.setup_styles()
        
        # Creazione dell'interfaccia
        self.create_widgets()
        
        # Binding degli eventi
        self.brightness_var.trace_add('write', self.apply_adjustments)
        self.contrast_var.trace_add('write', self.apply_adjustments)
        self.temperature_var.trace_add('write', self.apply_adjustments)
        
    def setup_styles(self):
        """Configura gli stili moderni per l'interfaccia"""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colori moderni
        bg_dark = "#1a1a1a"
        bg_medium = "#2d2d2d"
        bg_light = "#3d3d3d"
        accent = "#4a9eff"
        text_color = "#ffffff"
        
        # Stile per i pulsanti
        style.configure('Modern.TButton',
                       background=bg_medium,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10))
        style.map('Modern.TButton',
                 background=[('active', bg_light), ('pressed', accent)])
        
        # Stile per i pulsanti attivi (filtri applicati)
        style.configure('Active.TButton',
                       background=accent,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10, 'bold'))
        style.map('Active.TButton',
                 background=[('active', '#3a8eef'), ('pressed', '#2a7edf')])
        
        # Stile per le etichette
        style.configure('Modern.TLabel',
                       background=bg_dark,
                       foreground=text_color,
                       font=('Segoe UI', 10))
        
        # Stile per i frame
        style.configure('Modern.TFrame',
                       background=bg_dark)
        
        # Stile per gli slider
        style.configure('Modern.Horizontal.TScale',
                       background=bg_dark,
                       troughcolor=bg_medium,
                       borderwidth=0,
                       sliderthickness=20)
        
    def create_widgets(self):
        """Crea tutti i widget dell'interfaccia"""
        # Frame principale
        main_frame = ttk.Frame(self.root, style='Modern.TFrame')
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Area superiore: canvas per l'immagine
        self.canvas_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        self.canvas_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 10))
        
        self.canvas = tk.Canvas(self.canvas_frame,
                               bg="#2d2d2d",
                               highlightthickness=0,
                               cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Testo placeholder
        self.canvas.create_text(
            400, 300,
            text="Carica un'immagine per iniziare",
            fill="#666666",
            font=('Segoe UI', 16),
            tags="placeholder"
        )
        
        # Binding per il crop
        self.canvas.bind("<Button-1>", self.on_crop_start)
        self.canvas.bind("<B1-Motion>", self.on_crop_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_end)
        
        # Area inferiore: controlli
        controls_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        controls_frame.pack(fill=tk.X)
        
        # Frame per i pulsanti principali - organizzato in righe per essere responsive
        buttons_container = ttk.Frame(controls_frame, style='Modern.TFrame')
        buttons_container.pack(fill=tk.X, pady=(0, 10))
        
        # Prima riga: Caricamento e Salvataggio
        row1_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row1_frame.pack(fill=tk.X, pady=2)
        
        load_btn = ttk.Button(row1_frame,
                             text="📁 Carica Immagine",
                             command=self.load_image,
                             style='Modern.TButton')
        load_btn.pack(side=tk.LEFT, padx=5)
        
        save_jpg_btn = ttk.Button(row1_frame,
                                 text="💾 Salva JPG",
                                 command=lambda: self.save_image('jpg'),
                                 style='Modern.TButton')
        save_jpg_btn.pack(side=tk.LEFT, padx=5)
        
        save_png_btn = ttk.Button(row1_frame,
                                 text="💾 Salva PNG",
                                 command=lambda: self.save_image('png'),
                                 style='Modern.TButton')
        save_png_btn.pack(side=tk.LEFT, padx=5)
        
        # Seconda riga: Filtri
        row2_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row2_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row2_frame, text="Filtri:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        self.vintage_btn = ttk.Button(row2_frame,
                                text="🎨 Vintage",
                                command=self.toggle_vintage,
                                style='Modern.TButton')
        self.vintage_btn.pack(side=tk.LEFT, padx=5)
        
        self.bw_btn = ttk.Button(row2_frame,
                           text="⚫ Bianco e Nero",
                           command=self.toggle_bw,
                           style='Modern.TButton')
        self.bw_btn.pack(side=tk.LEFT, padx=5)
        
        # Terza riga: Trasformazioni
        row3_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row3_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row3_frame, text="Trasformazioni:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        crop_btn = ttk.Button(row3_frame,
                             text="✂️ Ritaglia",
                             command=self.toggle_crop_mode,
                             style='Modern.TButton')
        crop_btn.pack(side=tk.LEFT, padx=5)
        
        mirror_btn = ttk.Button(row3_frame,
                               text="🔄 Specchio",
                               command=self.apply_mirror,
                               style='Modern.TButton')
        mirror_btn.pack(side=tk.LEFT, padx=5)
        
        rotate_btn = ttk.Button(row3_frame,
                               text="🔃 Ruota 90°",
                               command=self.apply_rotation,
                               style='Modern.TButton')
        rotate_btn.pack(side=tk.LEFT, padx=5)
        
        # Quarta riga: Undo e Reset
        row4_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row4_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row4_frame, text="Controlli:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        undo_btn = ttk.Button(row4_frame,
                             text="⮌ Undo",
                             command=self.undo,
                             style='Modern.TButton')
        undo_btn.pack(side=tk.LEFT, padx=5)
        
        reset_btn = ttk.Button(row4_frame,
                              text="↺ Reset",
                              command=self.reset_image,
                              style='Modern.TButton')
        reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Frame per gli slider
        sliders_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        sliders_frame.pack(fill=tk.X)
        
        # Slider luminosità
        self.create_slider(sliders_frame, "☀️ Luminosità", self.brightness_var, 0.0, 2.0, 0)
        
        # Slider contrasto
        self.create_slider(sliders_frame, "◐ Contrasto", self.contrast_var, 0.0, 2.0, 1)
        
        # Slider temperatura
        self.create_slider(sliders_frame, "🌡️ Temperatura", self.temperature_var, -100, 100, 2)
        
    def create_slider(self, parent, label_text, variable, from_, to, column):
        """Crea uno slider con etichetta"""
        frame = ttk.Frame(parent, style='Modern.TFrame')
        frame.grid(row=0, column=column, padx=10, pady=5, sticky='ew')
        parent.columnconfigure(column, weight=1)
        
        label = ttk.Label(frame, text=label_text, style='Modern.TLabel')
        label.pack(anchor='w')
        
        slider = ttk.Scale(frame,
                          from_=from_,
                          to=to,
                          variable=variable,
                          orient=tk.HORIZONTAL,
                          style='Modern.Horizontal.TScale')
        slider.pack(fill=tk.X, pady=5)
        
        # Etichetta valore
        value_label = ttk.Label(frame, text=f"{variable.get():.2f}", style='Modern.TLabel')
        value_label.pack(anchor='e')
        
        def update_label(*args):
            value_label.config(text=f"{variable.get():.2f}")
        
        variable.trace_add('write', update_label)
        
    def load_image(self):
        """Carica un'immagine dal file system"""
        file_path = filedialog.askopenfilename(
            title="Seleziona un'immagine",
            filetypes=[
                ("Immagini", "*.jpg *.jpeg *.png *.bmp *.gif"),
                ("JPEG", "*.jpg *.jpeg"),
                ("PNG", "*.png"),
                ("Tutti i file", "*.*")
            ]
        )
        
        if file_path:
            try:
                self.original_image = Image.open(file_path)
                self.current_image = self.original_image.copy()
                
                # Reset di tutti i controlli e filtri
                self.brightness_var.set(1.0)
                self.contrast_var.set(1.0)
                self.temperature_var.set(0)
                
                # Reset stato filtri
                self.vintage_active = False
                self.bw_active = False
                self.update_filter_buttons()
                
                # Reset cronologia
                self.history = []
                self.save_to_history()
                
                self.display_current_image()
                self.canvas.delete("placeholder")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile caricare l'immagine:\n{str(e)}")
                
    def display_current_image(self):
        """Visualizza l'immagine corrente sul canvas"""
        if self.current_image is None:
            return
            
        # Calcola le dimensioni per il fit
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
            
        # Calcola il ridimensionamento mantenendo l'aspect ratio
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        new_width = int(img_width * ratio * 0.9)  # 90% per margini
        new_height = int(img_height * ratio * 0.9)
        
        # Ridimensiona l'immagine
        display_img = self.current_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Converti in PhotoImage
        self.display_image = ImageTk.PhotoImage(display_img)
        
        # Pulisci il canvas e mostra l'immagine
        self.canvas.delete("all")
        self.canvas.create_image(
            canvas_width // 2,
            canvas_height // 2,
            image=self.display_image,
            anchor=tk.CENTER,
            tags="image"
        )
        
    def save_to_history(self):
        """Salva lo stato corrente nella cronologia"""
        if self.original_image is None:
            return
            
        # Salva una copia dell'immagine e dello stato dei filtri
        state = {
            'image': self.original_image.copy(),
            'brightness': self.brightness_var.get(),
            'contrast': self.contrast_var.get(),
            'temperature': self.temperature_var.get(),
            'vintage': self.vintage_active,
            'bw': self.bw_active
        }
        
        self.history.append(state)
        
        # Limita la dimensione della cronologia
        if len(self.history) > self.max_history:
            self.history.pop(0)
            
    def undo(self):
        """Annulla l'ultima modifica"""
        if len(self.history) <= 1:
            messagebox.showinfo("Undo", "Nessuna operazione da annullare!")
            return
            
        # Rimuovi lo stato corrente e torna al precedente
        self.history.pop()
        
        # Ripristina lo stato precedente
        if self.history:
            previous_state = self.history[-1]
            self.original_image = previous_state['image'].copy()
            self.brightness_var.set(previous_state['brightness'])
            self.contrast_var.set(previous_state['contrast'])
            self.temperature_var.set(previous_state['temperature'])
            self.vintage_active = previous_state['vintage']
            self.bw_active = previous_state['bw']
            
            self.update_filter_buttons()
            self.apply_adjustments()
            
            # NON salvare in cronologia dopo undo - questo era il bug!
            # Rimuovi l'ultimo salvataggio che apply_adjustments ha fatto
            if len(self.history) > 1 and self.history[-1] == self.history[-2]:
                self.history.pop()
            
    def update_filter_buttons(self):
        """Aggiorna lo stile dei pulsanti filtro in base allo stato"""
        if self.vintage_btn:
            style = 'Active.TButton' if self.vintage_active else 'Modern.TButton'
            self.vintage_btn.configure(style=style)
            
        if self.bw_btn:
            style = 'Active.TButton' if self.bw_active else 'Modern.TButton'
            self.bw_btn.configure(style=style)
    
    def get_base_image(self):
        """Ottiene l'immagine base senza filtri cromatici dalla cronologia"""
        # Cerca l'ultima immagine senza filtri vintage o b/n
        for i in range(len(self.history) - 1, -1, -1):
            state = self.history[i]
            if not state['vintage'] and not state['bw']:
                return state['image'].copy()
        
        # Se non trovata, usa la prima immagine della cronologia
        if self.history:
            return self.history[0]['image'].copy()
        
        # Fallback (non dovrebbe mai succedere)
        return self.original_image.copy()
        
    def apply_adjustments(self, *args):
        """Applica le regolazioni di luminosità, contrasto e temperatura"""
        if self.original_image is None:
            return
            
        # Parte dall'immagine originale
        img = self.original_image.copy()
        
        # Applica luminosità
        if self.brightness_var.get() != 1.0:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(self.brightness_var.get())
            
        # Applica contrasto
        if self.contrast_var.get() != 1.0:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(self.contrast_var.get())
            
        # Applica temperatura (modifica del bilanciamento del colore)
        if self.temperature_var.get() != 0:
            img = self.adjust_temperature(img, self.temperature_var.get())
            
        self.current_image = img
        self.display_current_image()
        
    def adjust_temperature(self, image, value):
        """Regola la temperatura del colore dell'immagine"""
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        # Converti in array numpy
        img_array = np.array(image, dtype=np.float32)
        
        # Applica shift di temperatura
        if value > 0:  # Più caldo (più rosso/giallo)
            img_array[:, :, 0] += value * 0.5  # Rosso
            img_array[:, :, 1] += value * 0.3  # Verde
        else:  # Più freddo (più blu)
            img_array[:, :, 2] += abs(value) * 0.5  # Blu
            
        # Clamp values
        img_array = np.clip(img_array, 0, 255)
        
        return Image.fromarray(img_array.astype(np.uint8))
        
    def toggle_vintage(self):
        """Attiva/disattiva il filtro vintage (seppia)"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Salva lo stato corrente prima di modificare
        self.save_to_history()
        
        # Toggle dello stato
        self.vintage_active = not self.vintage_active
        
        # Se stiamo attivando vintage, disattiva B/N
        if self.vintage_active and self.bw_active:
            self.bw_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.vintage_active:
            # Applica filtro vintage sull'immagine base
            img = base_image.convert('RGB')
            img_array = np.array(img, dtype=np.float32)
            
            # Matrice seppia
            sepia_filter = np.array([
                [0.393, 0.769, 0.189],
                [0.349, 0.686, 0.168],
                [0.272, 0.534, 0.131]
            ])
            
            # Applica il filtro
            sepia_img = img_array @ sepia_filter.T
            sepia_img = np.clip(sepia_img, 0, 255)
            
            self.original_image = Image.fromarray(sepia_img.astype(np.uint8))
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        
    def toggle_bw(self):
        """Attiva/disattiva il filtro bianco e nero"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Salva lo stato corrente prima di modificare
        self.save_to_history()
        
        # Toggle dello stato
        self.bw_active = not self.bw_active
        
        # Se stiamo attivando B/N, disattiva Vintage
        if self.bw_active and self.vintage_active:
            self.vintage_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.bw_active:
            # Applica filtro bianco e nero sull'immagine base
            self.original_image = base_image.convert('L').convert('RGB')
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        
    def toggle_crop_mode(self):
        """Attiva/disattiva la modalità ritaglio"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.crop_mode = not self.crop_mode
        if self.crop_mode:
            self.canvas.config(cursor="cross")
            messagebox.showinfo("Modalità Ritaglio",
                              "Trascina il mouse sull'immagine per selezionare l'area da ritagliare.\n"
                              "Clicca di nuovo su 'Ritaglia' per applicare.")
        else:
            self.canvas.config(cursor="arrow")
            if self.crop_rect:
                self.apply_crop()
                
    def on_crop_start(self, event):
        """Inizia la selezione per il ritaglio"""
        if not self.crop_mode:
            return
        self.crop_start = (event.x, event.y)
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
    def on_crop_drag(self, event):
        """Disegna il rettangolo di selezione"""
        if not self.crop_mode or not self.crop_start:
            return
            
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
        self.crop_rect = self.canvas.create_rectangle(
            self.crop_start[0], self.crop_start[1],
            event.x, event.y,
            outline='#4a9eff',
            width=2,
            dash=(5, 5)
        )
        
    def on_crop_end(self, event):
        """Finalizza la selezione"""
        if not self.crop_mode or not self.crop_start:
            return
        self.crop_end = (event.x, event.y)
        
    def apply_crop(self):
        """Applica il ritaglio all'immagine"""
        if not self.crop_start or not self.crop_end:
            return
            
        # Calcola le coordinate relative all'immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Converti coordinate canvas in coordinate immagine
        x1 = int((min(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y1 = int((min(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        x2 = int((max(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y2 = int((max(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        
        # Clamp ai limiti dell'immagine
        x1 = max(0, min(x1, img_width))
        y1 = max(0, min(y1, img_height))
        x2 = max(0, min(x2, img_width))
        y2 = max(0, min(y2, img_height))
        
        if x2 > x1 and y2 > y1:
            self.save_to_history()
            self.original_image = self.current_image.crop((x1, y1, x2, y2))
            self.apply_adjustments()
            
        self.crop_mode = False
        self.crop_start = None
        self.crop_end = None
        self.crop_rect = None
        self.canvas.config(cursor="arrow")
        
    def apply_mirror(self):
        """Applica l'effetto specchio (flip orizzontale)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.save_to_history()
        self.original_image = self.current_image.transpose(Image.FLIP_LEFT_RIGHT)
        self.apply_adjustments()
        
    def apply_rotation(self):
        """Ruota l'immagine di 90 gradi in senso orario"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.save_to_history()
        self.original_image = self.current_image.rotate(-90, expand=True)
        self.apply_adjustments()
        
    def reset_image(self):
        """Resetta tutte le modifiche e torna all'immagine originale"""
        if not self.history:
            return
        
        # Torna alla primissima immagine caricata
        first_state = self.history[0]
        self.original_image = first_state['image'].copy()
        
        # Reset di tutti i controlli
        self.brightness_var.set(1.0)
        self.contrast_var.set(1.0)
        self.temperature_var.set(0)
        
        # Reset di tutti i filtri
        self.vintage_active = False
        self.bw_active = False
        self.update_filter_buttons()
        
        # Resetta la cronologia mantenendo solo lo stato originale
        self.history = [first_state]
        
        self.apply_adjustments()
        
    def save_image(self, format_type):
        """Salva l'immagine nel formato specificato"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Nessuna immagine da salvare!")
            return
            
        file_extension = f".{format_type}"
        file_types = [(format_type.upper(), f"*{file_extension}")]
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=file_extension,
            filetypes=file_types,
            title=f"Salva come {format_type.upper()}"
        )
        
        if file_path:
            try:
                if format_type.lower() == 'jpg':
                    # Converti in RGB se necessario (JPG non supporta trasparenza)
                    img_to_save = self.current_image
                    if img_to_save.mode in ('RGBA', 'LA', 'P'):
                        rgb_img = Image.new('RGB', img_to_save.size, (255, 255, 255))
                        if img_to_save.mode == 'P':
                            img_to_save = img_to_save.convert('RGBA')
                        rgb_img.paste(img_to_save, mask=img_to_save.split()[-1] if img_to_save.mode == 'RGBA' else None)
                        img_to_save = rgb_img
                    img_to_save.save(file_path, 'JPEG', quality=95)
                else:
                    self.current_image.save(file_path, 'PNG')
                    
                messagebox.showinfo("Successo", f"Immagine salvata con successo in:\n{file_path}")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile salvare l'immagine:\n{str(e)}")


def main():
    """Funzione principale"""
    root = tk.Tk()
    app = PhotoEditor(root)
    
    # Configura il ridimensionamento della finestra
    root.minsize(800, 600)
    
    # Centra la finestra
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()


## Prompt 4

Va risolto il problema dell'Undo.     
Quando devo annullare 1 operazione, la annulla correttamente, quando devo annullare n operazioni e clicco n volte 'Undo', l'immagine si resetta sempre dopo n-1 click, perdendosi a tutti gli effetti una modifica.      
Inoltre, all'apertura, la scritta iniziale sullo sfondo 'Carica un'immagine' è leggermente decentrata a sinistra. Vorrei averla centrata.    
Un'altra modifica che vorrei è l'aggiunta del pulsante 'Testo' su Trasformazioni, che mi permette di aggiungere una casella di testo mobile, modificabile dall'utente in posizione, grandezza e colore.      

In [2]:
"""
Editor di Foto Professionale
Applicazione moderna per la modifica di immagini con interfaccia intuitiva
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk, ImageEnhance, ImageFilter, ImageOps
import numpy as np
from typing import Optional
import os


class PhotoEditor:
    """Editor di foto con interfaccia grafica moderna"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Editor di Foto Professionale")
        self.root.geometry("1200x800")
        self.root.configure(bg="#1a1a1a")
        
        # Variabili per le immagini
        self.original_image: Optional[Image.Image] = None
        self.current_image: Optional[Image.Image] = None
        self.display_image: Optional[ImageTk.PhotoImage] = None
        self.crop_mode = False
        self.crop_start = None
        self.crop_rect = None
        
        # Cronologia per Undo (stack di immagini)
        self.history = []
        self.max_history = 20  # Massimo 20 stati nella cronologia
        
        # Variabili per i filtri
        self.brightness_var = tk.DoubleVar(value=1.0)
        self.contrast_var = tk.DoubleVar(value=1.0)
        self.temperature_var = tk.DoubleVar(value=0)
        
        # Stati dei filtri toggle
        self.vintage_active = False
        self.bw_active = False
        
        # Riferimenti ai pulsanti per cambio colore
        self.vintage_btn = None
        self.bw_btn = None
        
        # Gestione testi sull'immagine
        self.text_items = []  # Lista di dizionari con info sui testi
        self.selected_text = None
        self.text_drag_data = {"x": 0, "y": 0}
        
        # Configurazione dello stile moderno
        self.setup_styles()
        
        # Creazione dell'interfaccia
        self.create_widgets()
        
        # Binding degli eventi e salva gli ID dei trace
        self.brightness_trace_id = self.brightness_var.trace_add('write', self.apply_adjustments)
        self.contrast_trace_id = self.contrast_var.trace_add('write', self.apply_adjustments)
        self.temperature_trace_id = self.temperature_var.trace_add('write', self.apply_adjustments)
        
    def setup_styles(self):
        """Configura gli stili moderni per l'interfaccia"""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colori moderni
        bg_dark = "#1a1a1a"
        bg_medium = "#2d2d2d"
        bg_light = "#3d3d3d"
        accent = "#4a9eff"
        text_color = "#ffffff"
        
        # Stile per i pulsanti
        style.configure('Modern.TButton',
                       background=bg_medium,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10))
        style.map('Modern.TButton',
                 background=[('active', bg_light), ('pressed', accent)])
        
        # Stile per i pulsanti attivi (filtri applicati)
        style.configure('Active.TButton',
                       background=accent,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10, 'bold'))
        style.map('Active.TButton',
                 background=[('active', '#3a8eef'), ('pressed', '#2a7edf')])
        
        # Stile per le etichette
        style.configure('Modern.TLabel',
                       background=bg_dark,
                       foreground=text_color,
                       font=('Segoe UI', 10))
        
        # Stile per i frame
        style.configure('Modern.TFrame',
                       background=bg_dark)
        
        # Stile per gli slider
        style.configure('Modern.Horizontal.TScale',
                       background=bg_dark,
                       troughcolor=bg_medium,
                       borderwidth=0,
                       sliderthickness=20)
        
    def create_widgets(self):
        """Crea tutti i widget dell'interfaccia"""
        # Frame principale
        main_frame = ttk.Frame(self.root, style='Modern.TFrame')
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Area superiore: canvas per l'immagine
        self.canvas_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        self.canvas_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 10))
        
        self.canvas = tk.Canvas(self.canvas_frame,
                               bg="#2d2d2d",
                               highlightthickness=0,
                               cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Salva il riferimento al placeholder
        self.placeholder_text = None
        
        # Crea il placeholder dopo che il canvas è stato disegnato
        self.canvas.update_idletasks()
        self.create_placeholder()
        
        # Binding per il crop
        self.canvas.bind("<Button-1>", self.on_crop_start)
        self.canvas.bind("<B1-Motion>", self.on_crop_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_end)
        
        # Binding per ridimensionamento finestra
        self.canvas.bind("<Configure>", self.on_canvas_resize)
        
        # Area inferiore: controlli
        controls_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        controls_frame.pack(fill=tk.X)
        
        # Frame per i pulsanti principali - organizzato in righe per essere responsive
        buttons_container = ttk.Frame(controls_frame, style='Modern.TFrame')
        buttons_container.pack(fill=tk.X, pady=(0, 10))
        
        # Prima riga: Caricamento e Salvataggio
        row1_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row1_frame.pack(fill=tk.X, pady=2)
        
        load_btn = ttk.Button(row1_frame,
                             text="📁 Carica Immagine",
                             command=self.load_image,
                             style='Modern.TButton')
        load_btn.pack(side=tk.LEFT, padx=5)
        
        save_jpg_btn = ttk.Button(row1_frame,
                                 text="💾 Salva JPG",
                                 command=lambda: self.save_image('jpg'),
                                 style='Modern.TButton')
        save_jpg_btn.pack(side=tk.LEFT, padx=5)
        
        save_png_btn = ttk.Button(row1_frame,
                                 text="💾 Salva PNG",
                                 command=lambda: self.save_image('png'),
                                 style='Modern.TButton')
        save_png_btn.pack(side=tk.LEFT, padx=5)
        
        # Seconda riga: Filtri
        row2_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row2_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row2_frame, text="Filtri:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        self.vintage_btn = ttk.Button(row2_frame,
                                text="🎨 Vintage",
                                command=self.toggle_vintage,
                                style='Modern.TButton')
        self.vintage_btn.pack(side=tk.LEFT, padx=5)
        
        self.bw_btn = ttk.Button(row2_frame,
                           text="⚫ Bianco e Nero",
                           command=self.toggle_bw,
                           style='Modern.TButton')
        self.bw_btn.pack(side=tk.LEFT, padx=5)
        
        # Terza riga: Trasformazioni
        row3_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row3_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row3_frame, text="Trasformazioni:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        crop_btn = ttk.Button(row3_frame,
                             text="✂️ Ritaglia",
                             command=self.toggle_crop_mode,
                             style='Modern.TButton')
        crop_btn.pack(side=tk.LEFT, padx=5)
        
        mirror_btn = ttk.Button(row3_frame,
                               text="🔄 Specchio",
                               command=self.apply_mirror,
                               style='Modern.TButton')
        mirror_btn.pack(side=tk.LEFT, padx=5)
        
        rotate_btn = ttk.Button(row3_frame,
                               text="🔃 Ruota 90°",
                               command=self.apply_rotation,
                               style='Modern.TButton')
        rotate_btn.pack(side=tk.LEFT, padx=5)
        
        text_btn = ttk.Button(row3_frame,
                             text="📝 Testo",
                             command=self.add_text,
                             style='Modern.TButton')
        text_btn.pack(side=tk.LEFT, padx=5)
        
        # Quarta riga: Undo e Reset
        row4_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row4_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row4_frame, text="Controlli:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        undo_btn = ttk.Button(row4_frame,
                             text="⮌ Undo",
                             command=self.undo,
                             style='Modern.TButton')
        undo_btn.pack(side=tk.LEFT, padx=5)
        
        reset_btn = ttk.Button(row4_frame,
                              text="↺ Reset",
                              command=self.reset_image,
                              style='Modern.TButton')
        reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Frame per gli slider
        sliders_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        sliders_frame.pack(fill=tk.X)
        
        # Slider luminosità
        self.create_slider(sliders_frame, "☀️ Luminosità", self.brightness_var, 0.0, 2.0, 0)
        
        # Slider contrasto
        self.create_slider(sliders_frame, "◐ Contrasto", self.contrast_var, 0.0, 2.0, 1)
        
        # Slider temperatura
        self.create_slider(sliders_frame, "🌡️ Temperatura", self.temperature_var, -100, 100, 2)
        
    def create_slider(self, parent, label_text, variable, from_, to, column):
        """Crea uno slider con etichetta"""
        frame = ttk.Frame(parent, style='Modern.TFrame')
        frame.grid(row=0, column=column, padx=10, pady=5, sticky='ew')
        parent.columnconfigure(column, weight=1)
        
        label = ttk.Label(frame, text=label_text, style='Modern.TLabel')
        label.pack(anchor='w')
        
        slider = ttk.Scale(frame,
                          from_=from_,
                          to=to,
                          variable=variable,
                          orient=tk.HORIZONTAL,
                          style='Modern.Horizontal.TScale')
        slider.pack(fill=tk.X, pady=5)
        
        # Etichetta valore
        value_label = ttk.Label(frame, text=f"{variable.get():.2f}", style='Modern.TLabel')
        value_label.pack(anchor='e')
        
        def update_label(*args):
            value_label.config(text=f"{variable.get():.2f}")
        
        variable.trace_add('write', update_label)
        
    def load_image(self):
        """Carica un'immagine dal file system"""
        file_path = filedialog.askopenfilename(
            title="Seleziona un'immagine",
            filetypes=[
                ("Immagini", "*.jpg *.jpeg *.png *.bmp *.gif"),
                ("JPEG", "*.jpg *.jpeg"),
                ("PNG", "*.png"),
                ("Tutti i file", "*.*")
            ]
        )
        
        if file_path:
            try:
                self.original_image = Image.open(file_path)
                self.current_image = self.original_image.copy()
                
                # Reset di tutti i controlli e filtri
                self.brightness_var.set(1.0)
                self.contrast_var.set(1.0)
                self.temperature_var.set(0)
                
                # Reset stato filtri
                self.vintage_active = False
                self.bw_active = False
                self.update_filter_buttons()
                
                # Reset cronologia
                self.history = []
                self.save_to_history()
                
                self.display_current_image()
                self.canvas.delete("placeholder")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile caricare l'immagine:\n{str(e)}")
                
    def display_current_image(self):
        """Visualizza l'immagine corrente sul canvas"""
        if self.current_image is None:
            return
            
        # Calcola le dimensioni per il fit
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
            
        # Calcola il ridimensionamento mantenendo l'aspect ratio
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        new_width = int(img_width * ratio * 0.9)  # 90% per margini
        new_height = int(img_height * ratio * 0.9)
        
        # Ridimensiona l'immagine
        display_img = self.current_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Converti in PhotoImage
        self.display_image = ImageTk.PhotoImage(display_img)
        
        # Pulisci il canvas e mostra l'immagine
        self.canvas.delete("all")
        self.canvas.create_image(
            canvas_width // 2,
            canvas_height // 2,
            image=self.display_image,
            anchor=tk.CENTER,
            tags="image"
        )
        
    def save_to_history(self):
        """Salva lo stato corrente nella cronologia"""
        if self.original_image is None:
            return
            
        # Salva una copia dell'immagine e dello stato dei filtri
        state = {
            'image': self.original_image.copy(),
            'brightness': self.brightness_var.get(),
            'contrast': self.contrast_var.get(),
            'temperature': self.temperature_var.get(),
            'vintage': self.vintage_active,
            'bw': self.bw_active
        }
        
        self.history.append(state)
        
        # Limita la dimensione della cronologia
        if len(self.history) > self.max_history:
            self.history.pop(0)
            
    def undo(self):
        """Annulla l'ultima modifica"""
        if len(self.history) <= 1:
            messagebox.showinfo("Undo", "Nessuna operazione da annullare!")
            return
            
        # Rimuovi lo stato corrente e torna al precedente
        self.history.pop()
        
        # Ripristina lo stato precedente
        if self.history:
            previous_state = self.history[-1]
            self.original_image = previous_state['image'].copy()
            
            # Disattiva temporaneamente i trace per evitare salvataggi in cronologia
            self.brightness_var.trace_remove('write', self.brightness_trace_id)
            self.contrast_var.trace_remove('write', self.contrast_trace_id)
            self.temperature_var.trace_remove('write', self.temperature_trace_id)
            
            self.brightness_var.set(previous_state['brightness'])
            self.contrast_var.set(previous_state['contrast'])
            self.temperature_var.set(previous_state['temperature'])
            
            # Riattiva i trace
            self.brightness_trace_id = self.brightness_var.trace_add('write', self.apply_adjustments)
            self.contrast_trace_id = self.contrast_var.trace_add('write', self.apply_adjustments)
            self.temperature_trace_id = self.temperature_var.trace_add('write', self.apply_adjustments)
            
            self.vintage_active = previous_state['vintage']
            self.bw_active = previous_state['bw']
            
            self.update_filter_buttons()
            self.apply_adjustments()
            
    def update_filter_buttons(self):
        """Aggiorna lo stile dei pulsanti filtro in base allo stato"""
        if self.vintage_btn:
            style = 'Active.TButton' if self.vintage_active else 'Modern.TButton'
            self.vintage_btn.configure(style=style)
            
        if self.bw_btn:
            style = 'Active.TButton' if self.bw_active else 'Modern.TButton'
            self.bw_btn.configure(style=style)
    
    def create_placeholder(self):
        """Crea il testo placeholder centrato nel canvas"""
        if self.placeholder_text:
            self.canvas.delete(self.placeholder_text)
        
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        # Usa dimensioni di default se il canvas non è ancora renderizzato
        if canvas_width <= 1:
            canvas_width = 800
        if canvas_height <= 1:
            canvas_height = 600
        
        self.placeholder_text = self.canvas.create_text(
            canvas_width // 2,
            canvas_height // 2,
            text="Carica un'immagine per iniziare",
            fill="#666666",
            font=('Segoe UI', 16),
            tags="placeholder"
        )
    
    def on_canvas_resize(self, event):
        """Gestisce il ridimensionamento del canvas"""
        if self.original_image is None and self.placeholder_text:
            # Riposiziona il placeholder al centro
            self.create_placeholder()
        elif self.current_image is not None:
            # Ridisegna l'immagine per adattarla alle nuove dimensioni
            self.display_current_image()
    
    def get_base_image(self):
        """Ottiene l'immagine base senza filtri cromatici dalla cronologia"""
        # Cerca l'ultima immagine senza filtri vintage o b/n
        for i in range(len(self.history) - 1, -1, -1):
            state = self.history[i]
            if not state['vintage'] and not state['bw']:
                return state['image'].copy()
        
        # Se non trovata, usa la prima immagine della cronologia
        if self.history:
            return self.history[0]['image'].copy()
        
        # Fallback (non dovrebbe mai succedere)
        return self.original_image.copy()
        
    def apply_adjustments(self, *args):
        """Applica le regolazioni di luminosità, contrasto e temperatura"""
        if self.original_image is None:
            return
            
        # Parte dall'immagine originale
        img = self.original_image.copy()
        
        # Applica luminosità
        if self.brightness_var.get() != 1.0:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(self.brightness_var.get())
            
        # Applica contrasto
        if self.contrast_var.get() != 1.0:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(self.contrast_var.get())
            
        # Applica temperatura (modifica del bilanciamento del colore)
        if self.temperature_var.get() != 0:
            img = self.adjust_temperature(img, self.temperature_var.get())
            
        self.current_image = img
        self.display_current_image()
        
    def adjust_temperature(self, image, value):
        """Regola la temperatura del colore dell'immagine"""
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        # Converti in array numpy
        img_array = np.array(image, dtype=np.float32)
        
        # Applica shift di temperatura
        if value > 0:  # Più caldo (più rosso/giallo)
            img_array[:, :, 0] += value * 0.5  # Rosso
            img_array[:, :, 1] += value * 0.3  # Verde
        else:  # Più freddo (più blu)
            img_array[:, :, 2] += abs(value) * 0.5  # Blu
            
        # Clamp values
        img_array = np.clip(img_array, 0, 255)
        
        return Image.fromarray(img_array.astype(np.uint8))
        
    def toggle_vintage(self):
        """Attiva/disattiva il filtro vintage (seppia)"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Salva lo stato corrente prima di modificare
        self.save_to_history()
        
        # Toggle dello stato
        self.vintage_active = not self.vintage_active
        
        # Se stiamo attivando vintage, disattiva B/N
        if self.vintage_active and self.bw_active:
            self.bw_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.vintage_active:
            # Applica filtro vintage sull'immagine base
            img = base_image.convert('RGB')
            img_array = np.array(img, dtype=np.float32)
            
            # Matrice seppia
            sepia_filter = np.array([
                [0.393, 0.769, 0.189],
                [0.349, 0.686, 0.168],
                [0.272, 0.534, 0.131]
            ])
            
            # Applica il filtro
            sepia_img = img_array @ sepia_filter.T
            sepia_img = np.clip(sepia_img, 0, 255)
            
            self.original_image = Image.fromarray(sepia_img.astype(np.uint8))
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        
    def toggle_bw(self):
        """Attiva/disattiva il filtro bianco e nero"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Salva lo stato corrente prima di modificare
        self.save_to_history()
        
        # Toggle dello stato
        self.bw_active = not self.bw_active
        
        # Se stiamo attivando B/N, disattiva Vintage
        if self.bw_active and self.vintage_active:
            self.vintage_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.bw_active:
            # Applica filtro bianco e nero sull'immagine base
            self.original_image = base_image.convert('L').convert('RGB')
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        
    def toggle_crop_mode(self):
        """Attiva/disattiva la modalità ritaglio"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.crop_mode = not self.crop_mode
        if self.crop_mode:
            self.canvas.config(cursor="cross")
            messagebox.showinfo("Modalità Ritaglio",
                              "Trascina il mouse sull'immagine per selezionare l'area da ritagliare.\n"
                              "Clicca di nuovo su 'Ritaglia' per applicare.")
        else:
            self.canvas.config(cursor="arrow")
            if self.crop_rect:
                self.apply_crop()
                
    def on_crop_start(self, event):
        """Inizia la selezione per il ritaglio"""
        if not self.crop_mode:
            return
        self.crop_start = (event.x, event.y)
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
    def on_crop_drag(self, event):
        """Disegna il rettangolo di selezione"""
        if not self.crop_mode or not self.crop_start:
            return
            
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
        self.crop_rect = self.canvas.create_rectangle(
            self.crop_start[0], self.crop_start[1],
            event.x, event.y,
            outline='#4a9eff',
            width=2,
            dash=(5, 5)
        )
        
    def on_crop_end(self, event):
        """Finalizza la selezione"""
        if not self.crop_mode or not self.crop_start:
            return
        self.crop_end = (event.x, event.y)
        
    def apply_crop(self):
        """Applica il ritaglio all'immagine"""
        if not self.crop_start or not self.crop_end:
            return
            
        # Calcola le coordinate relative all'immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Converti coordinate canvas in coordinate immagine
        x1 = int((min(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y1 = int((min(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        x2 = int((max(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y2 = int((max(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        
        # Clamp ai limiti dell'immagine
        x1 = max(0, min(x1, img_width))
        y1 = max(0, min(y1, img_height))
        x2 = max(0, min(x2, img_width))
        y2 = max(0, min(y2, img_height))
        
        if x2 > x1 and y2 > y1:
            self.save_to_history()
            self.original_image = self.current_image.crop((x1, y1, x2, y2))
            self.apply_adjustments()
            
        self.crop_mode = False
        self.crop_start = None
        self.crop_end = None
        self.crop_rect = None
        self.canvas.config(cursor="arrow")
        
    def apply_mirror(self):
        """Applica l'effetto specchio (flip orizzontale)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.save_to_history()
        self.original_image = self.current_image.transpose(Image.FLIP_LEFT_RIGHT)
        self.apply_adjustments()
        
    def apply_rotation(self):
        """Ruota l'immagine di 90 gradi in senso orario"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.save_to_history()
        self.original_image = self.current_image.rotate(-90, expand=True)
        self.apply_adjustments()
    
    def add_text(self):
        """Apre una finestra di dialogo per aggiungere testo all'immagine"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Crea finestra di dialogo
        dialog = tk.Toplevel(self.root)
        dialog.title("Aggiungi Testo")
        dialog.geometry("400x300")
        dialog.configure(bg="#2d2d2d")
        dialog.transient(self.root)
        dialog.grab_set()
        
        # Frame principale
        main_frame = ttk.Frame(dialog, style='Modern.TFrame', padding=20)
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Campo testo
        ttk.Label(main_frame, text="Testo:", style='Modern.TLabel').pack(anchor='w', pady=(0, 5))
        text_entry = tk.Text(main_frame, height=3, width=40, bg="#3d3d3d", fg="white", 
                            font=('Segoe UI', 10), insertbackground='white')
        text_entry.pack(fill=tk.X, pady=(0, 15))
        text_entry.insert('1.0', 'Il tuo testo qui')
        text_entry.focus()
        
        # Dimensione font
        ttk.Label(main_frame, text="Dimensione Font:", style='Modern.TLabel').pack(anchor='w', pady=(0, 5))
        size_var = tk.IntVar(value=40)
        size_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        size_frame.pack(fill=tk.X, pady=(0, 15))
        size_slider = ttk.Scale(size_frame, from_=10, to=200, variable=size_var, 
                               orient=tk.HORIZONTAL, style='Modern.Horizontal.TScale')
        size_slider.pack(side=tk.LEFT, fill=tk.X, expand=True)
        size_label = ttk.Label(size_frame, text=f"{size_var.get()}", style='Modern.TLabel', width=5)
        size_label.pack(side=tk.LEFT, padx=(10, 0))
        
        def update_size_label(*args):
            size_label.config(text=f"{size_var.get()}")
        size_var.trace_add('write', update_size_label)
        
        # Colore
        ttk.Label(main_frame, text="Colore:", style='Modern.TLabel').pack(anchor='w', pady=(0, 5))
        color_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        color_frame.pack(fill=tk.X, pady=(0, 15))
        
        color_var = tk.StringVar(value="#FFFFFF")
        colors = [
            ("#FFFFFF", "Bianco"), ("#000000", "Nero"), ("#FF0000", "Rosso"),
            ("#00FF00", "Verde"), ("#0000FF", "Blu"), ("#FFFF00", "Giallo"),
            ("#FF00FF", "Magenta"), ("#00FFFF", "Ciano")
        ]
        
        for i, (hex_color, name) in enumerate(colors):
            btn = tk.Button(color_frame, bg=hex_color, width=3, height=1,
                          command=lambda c=hex_color: color_var.set(c),
                          relief=tk.RAISED, borderwidth=2)
            btn.pack(side=tk.LEFT, padx=2)
        
        # Pulsanti azione
        button_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        button_frame.pack(fill=tk.X, pady=(10, 0))
        
        def on_ok():
            text_content = text_entry.get('1.0', 'end-1c').strip()
            if text_content:
                self.apply_text_to_image(text_content, size_var.get(), color_var.get())
                dialog.destroy()
            else:
                messagebox.showwarning("Attenzione", "Inserisci del testo!", parent=dialog)
        
        def on_cancel():
            dialog.destroy()
        
        ok_btn = ttk.Button(button_frame, text="Aggiungi", command=on_ok, style='Modern.TButton')
        ok_btn.pack(side=tk.LEFT, padx=5)
        
        cancel_btn = ttk.Button(button_frame, text="Annulla", command=on_cancel, style='Modern.TButton')
        cancel_btn.pack(side=tk.LEFT, padx=5)
        
        # Enter per confermare
        dialog.bind('<Return>', lambda e: on_ok())
        dialog.bind('<Escape>', lambda e: on_cancel())
    
    def apply_text_to_image(self, text, font_size, color):
        """Applica il testo all'immagine"""
        from PIL import ImageDraw, ImageFont
        
        self.save_to_history()
        
        # Crea una copia dell'immagine originale per disegnare il testo
        img_with_text = self.original_image.copy()
        draw = ImageDraw.Draw(img_with_text)
        
        # Prova a caricare un font di sistema, altrimenti usa il default
        try:
            # Prova diversi font comuni
            font_options = [
                'arial.ttf', 'Arial.ttf',
                '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                '/System/Library/Fonts/Helvetica.ttc',
                'C:\\Windows\\Fonts\\arial.ttf'
            ]
            font = None
            for font_path in font_options:
                try:
                    font = ImageFont.truetype(font_path, font_size)
                    break
                except:
                    continue
            
            if font is None:
                font = ImageFont.load_default()
        except:
            font = ImageFont.load_default()
        
        # Posiziona il testo al centro dell'immagine
        img_width, img_height = img_with_text.size
        
        # Calcola dimensioni testo
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        
        # Centra il testo
        x = (img_width - text_width) // 2
        y = (img_height - text_height) // 2
        
        # Disegna il testo con bordo per migliore visibilità
        outline_color = "#000000" if color == "#FFFFFF" else "#FFFFFF"
        for adj_x in range(-2, 3):
            for adj_y in range(-2, 3):
                if adj_x != 0 or adj_y != 0:
                    draw.text((x + adj_x, y + adj_y), text, font=font, fill=outline_color)
        
        # Disegna il testo principale
        draw.text((x, y), text, font=font, fill=color)
        
        self.original_image = img_with_text
        self.apply_adjustments()
        
        messagebox.showinfo("Successo", "Testo aggiunto all'immagine!")
        
    def reset_image(self):
        """Resetta tutte le modifiche e torna all'immagine originale"""
        if not self.history:
            return
        
        # Torna alla primissima immagine caricata
        first_state = self.history[0]
        self.original_image = first_state['image'].copy()
        
        # Reset di tutti i controlli
        self.brightness_var.set(1.0)
        self.contrast_var.set(1.0)
        self.temperature_var.set(0)
        
        # Reset di tutti i filtri
        self.vintage_active = False
        self.bw_active = False
        self.update_filter_buttons()
        
        # Resetta la cronologia mantenendo solo lo stato originale
        self.history = [first_state]
        
        self.apply_adjustments()
        
    def save_image(self, format_type):
        """Salva l'immagine nel formato specificato"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Nessuna immagine da salvare!")
            return
            
        file_extension = f".{format_type}"
        file_types = [(format_type.upper(), f"*{file_extension}")]
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=file_extension,
            filetypes=file_types,
            title=f"Salva come {format_type.upper()}"
        )
        
        if file_path:
            try:
                if format_type.lower() == 'jpg':
                    # Converti in RGB se necessario (JPG non supporta trasparenza)
                    img_to_save = self.current_image
                    if img_to_save.mode in ('RGBA', 'LA', 'P'):
                        rgb_img = Image.new('RGB', img_to_save.size, (255, 255, 255))
                        if img_to_save.mode == 'P':
                            img_to_save = img_to_save.convert('RGBA')
                        rgb_img.paste(img_to_save, mask=img_to_save.split()[-1] if img_to_save.mode == 'RGBA' else None)
                        img_to_save = rgb_img
                    img_to_save.save(file_path, 'JPEG', quality=95)
                else:
                    self.current_image.save(file_path, 'PNG')
                    
                messagebox.showinfo("Successo", f"Immagine salvata con successo in:\n{file_path}")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile salvare l'immagine:\n{str(e)}")


def main():
    """Funzione principale"""
    root = tk.Tk()
    app = PhotoEditor(root)
    
    # Configura il ridimensionamento della finestra
    root.minsize(800, 600)
    
    # Centra la finestra
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()


## Prompt 5

Il problema dell'Undo persiste. Se faccio ad esempio 'Ruota'-'Specchio'-'B/N' poi clicco Undo una volta e mi toglie sia il B/N che lo specchio, clicco un'altra volta e mi toglie 'Ruota'. Perché accorpa sempre due operazioni in una e poi procede correttamente?      
Per quanto riguarda le problematiche del testo:       
- La finestra pop-up è troppo piccola all'apertura e non vedo bene tutti i bottoni, devo estenderla manualmente.      
- Nei pulsanti con i colori da Mac non vedo i colori, da Windows si. In ogni caso vorrei che quando seleziono un colore il bottone di quel colore diventa un po' piu grande in modo che l'utente possa capire quale bottone ha selezionato.      
- Il testo deve essere mobile e spostabile dal centro a piacimento dall'utente.       

In [3]:
"""
Editor di Foto Professionale
Applicazione moderna per la modifica di immagini con interfaccia intuitiva
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk, ImageEnhance, ImageFilter, ImageOps
import numpy as np
from typing import Optional
import os


class PhotoEditor:
    """Editor di foto con interfaccia grafica moderna"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Editor di Foto Professionale")
        self.root.geometry("1200x800")
        self.root.configure(bg="#1a1a1a")
        
        # Variabili per le immagini
        self.original_image: Optional[Image.Image] = None
        self.current_image: Optional[Image.Image] = None
        self.display_image: Optional[ImageTk.PhotoImage] = None
        self.crop_mode = False
        self.crop_start = None
        self.crop_rect = None
        
        # Cronologia per Undo (stack di immagini)
        self.history = []
        self.max_history = 20  # Massimo 20 stati nella cronologia
        
        # Variabili per i filtri
        self.brightness_var = tk.DoubleVar(value=1.0)
        self.contrast_var = tk.DoubleVar(value=1.0)
        self.temperature_var = tk.DoubleVar(value=0)
        
        # Stati dei filtri toggle
        self.vintage_active = False
        self.bw_active = False
        
        # Riferimenti ai pulsanti per cambio colore
        self.vintage_btn = None
        self.bw_btn = None
        
        # Gestione testi sull'immagine
        self.text_items = []  # Lista di dizionari con info sui testi
        self.selected_text = None
        self.text_drag_data = {"x": 0, "y": 0}
        
        # Configurazione dello stile moderno
        self.setup_styles()
        
        # Creazione dell'interfaccia
        self.create_widgets()
        
        # Binding degli eventi e salva gli ID dei trace
        self.brightness_trace_id = self.brightness_var.trace_add('write', self.apply_adjustments)
        self.contrast_trace_id = self.contrast_var.trace_add('write', self.apply_adjustments)
        self.temperature_trace_id = self.temperature_var.trace_add('write', self.apply_adjustments)
        
    def setup_styles(self):
        """Configura gli stili moderni per l'interfaccia"""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colori moderni
        bg_dark = "#1a1a1a"
        bg_medium = "#2d2d2d"
        bg_light = "#3d3d3d"
        accent = "#4a9eff"
        text_color = "#ffffff"
        
        # Stile per i pulsanti
        style.configure('Modern.TButton',
                       background=bg_medium,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10))
        style.map('Modern.TButton',
                 background=[('active', bg_light), ('pressed', accent)])
        
        # Stile per i pulsanti attivi (filtri applicati)
        style.configure('Active.TButton',
                       background=accent,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10, 'bold'))
        style.map('Active.TButton',
                 background=[('active', '#3a8eef'), ('pressed', '#2a7edf')])
        
        # Stile per le etichette
        style.configure('Modern.TLabel',
                       background=bg_dark,
                       foreground=text_color,
                       font=('Segoe UI', 10))
        
        # Stile per i frame
        style.configure('Modern.TFrame',
                       background=bg_dark)
        
        # Stile per gli slider
        style.configure('Modern.Horizontal.TScale',
                       background=bg_dark,
                       troughcolor=bg_medium,
                       borderwidth=0,
                       sliderthickness=20)
        
    def create_widgets(self):
        """Crea tutti i widget dell'interfaccia"""
        # Frame principale
        main_frame = ttk.Frame(self.root, style='Modern.TFrame')
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Area superiore: canvas per l'immagine
        self.canvas_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        self.canvas_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 10))
        
        self.canvas = tk.Canvas(self.canvas_frame,
                               bg="#2d2d2d",
                               highlightthickness=0,
                               cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Salva il riferimento al placeholder
        self.placeholder_text = None
        
        # Crea il placeholder dopo che il canvas è stato disegnato
        self.canvas.update_idletasks()
        self.create_placeholder()
        
        # Binding per il crop
        self.canvas.bind("<Button-1>", self.on_crop_start)
        self.canvas.bind("<B1-Motion>", self.on_crop_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_end)
        
        # Binding per ridimensionamento finestra
        self.canvas.bind("<Configure>", self.on_canvas_resize)
        
        # Area inferiore: controlli
        controls_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        controls_frame.pack(fill=tk.X)
        
        # Frame per i pulsanti principali - organizzato in righe per essere responsive
        buttons_container = ttk.Frame(controls_frame, style='Modern.TFrame')
        buttons_container.pack(fill=tk.X, pady=(0, 10))
        
        # Prima riga: Caricamento e Salvataggio
        row1_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row1_frame.pack(fill=tk.X, pady=2)
        
        load_btn = ttk.Button(row1_frame,
                             text="📁 Carica Immagine",
                             command=self.load_image,
                             style='Modern.TButton')
        load_btn.pack(side=tk.LEFT, padx=5)
        
        save_jpg_btn = ttk.Button(row1_frame,
                                 text="💾 Salva JPG",
                                 command=lambda: self.save_image('jpg'),
                                 style='Modern.TButton')
        save_jpg_btn.pack(side=tk.LEFT, padx=5)
        
        save_png_btn = ttk.Button(row1_frame,
                                 text="💾 Salva PNG",
                                 command=lambda: self.save_image('png'),
                                 style='Modern.TButton')
        save_png_btn.pack(side=tk.LEFT, padx=5)
        
        # Seconda riga: Filtri
        row2_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row2_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row2_frame, text="Filtri:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        self.vintage_btn = ttk.Button(row2_frame,
                                text="🎨 Vintage",
                                command=self.toggle_vintage,
                                style='Modern.TButton')
        self.vintage_btn.pack(side=tk.LEFT, padx=5)
        
        self.bw_btn = ttk.Button(row2_frame,
                           text="⚫ Bianco e Nero",
                           command=self.toggle_bw,
                           style='Modern.TButton')
        self.bw_btn.pack(side=tk.LEFT, padx=5)
        
        # Terza riga: Trasformazioni
        row3_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row3_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row3_frame, text="Trasformazioni:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        crop_btn = ttk.Button(row3_frame,
                             text="✂️ Ritaglia",
                             command=self.toggle_crop_mode,
                             style='Modern.TButton')
        crop_btn.pack(side=tk.LEFT, padx=5)
        
        mirror_btn = ttk.Button(row3_frame,
                               text="🔄 Specchio",
                               command=self.apply_mirror,
                               style='Modern.TButton')
        mirror_btn.pack(side=tk.LEFT, padx=5)
        
        rotate_btn = ttk.Button(row3_frame,
                               text="🔃 Ruota 90°",
                               command=self.apply_rotation,
                               style='Modern.TButton')
        rotate_btn.pack(side=tk.LEFT, padx=5)
        
        text_btn = ttk.Button(row3_frame,
                             text="📝 Testo",
                             command=self.add_text,
                             style='Modern.TButton')
        text_btn.pack(side=tk.LEFT, padx=5)
        
        # Quarta riga: Undo e Reset
        row4_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row4_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row4_frame, text="Controlli:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        undo_btn = ttk.Button(row4_frame,
                             text="⮌ Undo",
                             command=self.undo,
                             style='Modern.TButton')
        undo_btn.pack(side=tk.LEFT, padx=5)
        
        reset_btn = ttk.Button(row4_frame,
                              text="↺ Reset",
                              command=self.reset_image,
                              style='Modern.TButton')
        reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Frame per gli slider
        sliders_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        sliders_frame.pack(fill=tk.X)
        
        # Slider luminosità
        self.create_slider(sliders_frame, "☀️ Luminosità", self.brightness_var, 0.0, 2.0, 0)
        
        # Slider contrasto
        self.create_slider(sliders_frame, "◐ Contrasto", self.contrast_var, 0.0, 2.0, 1)
        
        # Slider temperatura
        self.create_slider(sliders_frame, "🌡️ Temperatura", self.temperature_var, -100, 100, 2)
        
    def create_slider(self, parent, label_text, variable, from_, to, column):
        """Crea uno slider con etichetta"""
        frame = ttk.Frame(parent, style='Modern.TFrame')
        frame.grid(row=0, column=column, padx=10, pady=5, sticky='ew')
        parent.columnconfigure(column, weight=1)
        
        label = ttk.Label(frame, text=label_text, style='Modern.TLabel')
        label.pack(anchor='w')
        
        slider = ttk.Scale(frame,
                          from_=from_,
                          to=to,
                          variable=variable,
                          orient=tk.HORIZONTAL,
                          style='Modern.Horizontal.TScale')
        slider.pack(fill=tk.X, pady=5)
        
        # Etichetta valore
        value_label = ttk.Label(frame, text=f"{variable.get():.2f}", style='Modern.TLabel')
        value_label.pack(anchor='e')
        
        def update_label(*args):
            value_label.config(text=f"{variable.get():.2f}")
        
        variable.trace_add('write', update_label)
        
    def load_image(self):
        """Carica un'immagine dal file system"""
        file_path = filedialog.askopenfilename(
            title="Seleziona un'immagine",
            filetypes=[
                ("Immagini", "*.jpg *.jpeg *.png *.bmp *.gif"),
                ("JPEG", "*.jpg *.jpeg"),
                ("PNG", "*.png"),
                ("Tutti i file", "*.*")
            ]
        )
        
        if file_path:
            try:
                self.original_image = Image.open(file_path)
                self.current_image = self.original_image.copy()
                
                # Reset di tutti i controlli e filtri
                self.brightness_var.set(1.0)
                self.contrast_var.set(1.0)
                self.temperature_var.set(0)
                
                # Reset stato filtri
                self.vintage_active = False
                self.bw_active = False
                self.update_filter_buttons()
                
                # Reset cronologia
                self.history = []
                self.save_to_history()
                
                self.display_current_image()
                self.canvas.delete("placeholder")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile caricare l'immagine:\n{str(e)}")
                
    def display_current_image(self):
        """Visualizza l'immagine corrente sul canvas"""
        if self.current_image is None:
            return
            
        # Calcola le dimensioni per il fit
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
            
        # Calcola il ridimensionamento mantenendo l'aspect ratio
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        new_width = int(img_width * ratio * 0.9)  # 90% per margini
        new_height = int(img_height * ratio * 0.9)
        
        # Ridimensiona l'immagine
        display_img = self.current_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Converti in PhotoImage
        self.display_image = ImageTk.PhotoImage(display_img)
        
        # Pulisci il canvas e mostra l'immagine
        self.canvas.delete("all")
        self.canvas.create_image(
            canvas_width // 2,
            canvas_height // 2,
            image=self.display_image,
            anchor=tk.CENTER,
            tags="image"
        )
        
    def save_to_history(self):
        """Salva lo stato corrente nella cronologia"""
        if self.original_image is None:
            return
            
        # Salva una copia dell'immagine e dello stato dei filtri
        state = {
            'image': self.original_image.copy(),
            'brightness': self.brightness_var.get(),
            'contrast': self.contrast_var.get(),
            'temperature': self.temperature_var.get(),
            'vintage': self.vintage_active,
            'bw': self.bw_active
        }
        
        self.history.append(state)
        
        # Limita la dimensione della cronologia
        if len(self.history) > self.max_history:
            self.history.pop(0)
            
    def undo(self):
        """Annulla l'ultima modifica"""
        if len(self.history) <= 1:
            messagebox.showinfo("Undo", "Nessuna operazione da annullare!")
            return
            
        # Rimuovi lo stato corrente e torna al precedente
        self.history.pop()
        
        # Ripristina lo stato precedente
        if self.history:
            previous_state = self.history[-1]
            self.original_image = previous_state['image'].copy()
            
            # Disattiva temporaneamente i trace per evitare salvataggi in cronologia
            self.brightness_var.trace_remove('write', self.brightness_trace_id)
            self.contrast_var.trace_remove('write', self.contrast_trace_id)
            self.temperature_var.trace_remove('write', self.temperature_trace_id)
            
            self.brightness_var.set(previous_state['brightness'])
            self.contrast_var.set(previous_state['contrast'])
            self.temperature_var.set(previous_state['temperature'])
            
            # Riattiva i trace
            self.brightness_trace_id = self.brightness_var.trace_add('write', self.apply_adjustments)
            self.contrast_trace_id = self.contrast_var.trace_add('write', self.apply_adjustments)
            self.temperature_trace_id = self.temperature_var.trace_add('write', self.apply_adjustments)
            
            self.vintage_active = previous_state['vintage']
            self.bw_active = previous_state['bw']
            
            self.update_filter_buttons()
            self.apply_adjustments()
            
    def update_filter_buttons(self):
        """Aggiorna lo stile dei pulsanti filtro in base allo stato"""
        if self.vintage_btn:
            style = 'Active.TButton' if self.vintage_active else 'Modern.TButton'
            self.vintage_btn.configure(style=style)
            
        if self.bw_btn:
            style = 'Active.TButton' if self.bw_active else 'Modern.TButton'
            self.bw_btn.configure(style=style)
    
    def create_placeholder(self):
        """Crea il testo placeholder centrato nel canvas"""
        if self.placeholder_text:
            self.canvas.delete(self.placeholder_text)
        
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        # Usa dimensioni di default se il canvas non è ancora renderizzato
        if canvas_width <= 1:
            canvas_width = 800
        if canvas_height <= 1:
            canvas_height = 600
        
        self.placeholder_text = self.canvas.create_text(
            canvas_width // 2,
            canvas_height // 2,
            text="Carica un'immagine per iniziare",
            fill="#666666",
            font=('Segoe UI', 16),
            tags="placeholder"
        )
    
    def on_canvas_resize(self, event):
        """Gestisce il ridimensionamento del canvas"""
        if self.original_image is None and self.placeholder_text:
            # Riposiziona il placeholder al centro
            self.create_placeholder()
        elif self.current_image is not None:
            # Ridisegna l'immagine per adattarla alle nuove dimensioni
            self.display_current_image()
    
    def get_base_image(self):
        """Ottiene l'immagine base senza filtri cromatici dalla cronologia"""
        # Cerca l'ultima immagine senza filtri vintage o b/n
        for i in range(len(self.history) - 1, -1, -1):
            state = self.history[i]
            if not state['vintage'] and not state['bw']:
                return state['image'].copy()
        
        # Se non trovata, usa la prima immagine della cronologia
        if self.history:
            return self.history[0]['image'].copy()
        
        # Fallback (non dovrebbe mai succedere)
        return self.original_image.copy()
        
    def apply_adjustments(self, *args):
        """Applica le regolazioni di luminosità, contrasto e temperatura"""
        if self.original_image is None:
            return
            
        # Parte dall'immagine originale
        img = self.original_image.copy()
        
        # Applica luminosità
        if self.brightness_var.get() != 1.0:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(self.brightness_var.get())
            
        # Applica contrasto
        if self.contrast_var.get() != 1.0:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(self.contrast_var.get())
            
        # Applica temperatura (modifica del bilanciamento del colore)
        if self.temperature_var.get() != 0:
            img = self.adjust_temperature(img, self.temperature_var.get())
            
        self.current_image = img
        self.display_current_image()
        
    def adjust_temperature(self, image, value):
        """Regola la temperatura del colore dell'immagine"""
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        # Converti in array numpy
        img_array = np.array(image, dtype=np.float32)
        
        # Applica shift di temperatura
        if value > 0:  # Più caldo (più rosso/giallo)
            img_array[:, :, 0] += value * 0.5  # Rosso
            img_array[:, :, 1] += value * 0.3  # Verde
        else:  # Più freddo (più blu)
            img_array[:, :, 2] += abs(value) * 0.5  # Blu
            
        # Clamp values
        img_array = np.clip(img_array, 0, 255)
        
        return Image.fromarray(img_array.astype(np.uint8))
        
    def toggle_vintage(self):
        """Attiva/disattiva il filtro vintage (seppia)"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Toggle dello stato
        self.vintage_active = not self.vintage_active
        
        # Se stiamo attivando vintage, disattiva B/N
        if self.vintage_active and self.bw_active:
            self.bw_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.vintage_active:
            # Applica filtro vintage sull'immagine base
            img = base_image.convert('RGB')
            img_array = np.array(img, dtype=np.float32)
            
            # Matrice seppia
            sepia_filter = np.array([
                [0.393, 0.769, 0.189],
                [0.349, 0.686, 0.168],
                [0.272, 0.534, 0.131]
            ])
            
            # Applica il filtro
            sepia_img = img_array @ sepia_filter.T
            sepia_img = np.clip(sepia_img, 0, 255)
            
            self.original_image = Image.fromarray(sepia_img.astype(np.uint8))
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
        
    def toggle_bw(self):
        """Attiva/disattiva il filtro bianco e nero"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Toggle dello stato
        self.bw_active = not self.bw_active
        
        # Se stiamo attivando B/N, disattiva Vintage
        if self.bw_active and self.vintage_active:
            self.vintage_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.bw_active:
            # Applica filtro bianco e nero sull'immagine base
            self.original_image = base_image.convert('L').convert('RGB')
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
        
    def toggle_crop_mode(self):
        """Attiva/disattiva la modalità ritaglio"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.crop_mode = not self.crop_mode
        if self.crop_mode:
            self.canvas.config(cursor="cross")
            messagebox.showinfo("Modalità Ritaglio",
                              "Trascina il mouse sull'immagine per selezionare l'area da ritagliare.\n"
                              "Clicca di nuovo su 'Ritaglia' per applicare.")
        else:
            self.canvas.config(cursor="arrow")
            if self.crop_rect:
                self.apply_crop()
                
    def on_crop_start(self, event):
        """Inizia la selezione per il ritaglio"""
        if not self.crop_mode:
            return
        self.crop_start = (event.x, event.y)
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
    def on_crop_drag(self, event):
        """Disegna il rettangolo di selezione"""
        if not self.crop_mode or not self.crop_start:
            return
            
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
        self.crop_rect = self.canvas.create_rectangle(
            self.crop_start[0], self.crop_start[1],
            event.x, event.y,
            outline='#4a9eff',
            width=2,
            dash=(5, 5)
        )
        
    def on_crop_end(self, event):
        """Finalizza la selezione"""
        if not self.crop_mode or not self.crop_start:
            return
        self.crop_end = (event.x, event.y)
        
    def apply_crop(self):
        """Applica il ritaglio all'immagine"""
        if not self.crop_start or not self.crop_end:
            return
            
        # Calcola le coordinate relative all'immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Converti coordinate canvas in coordinate immagine
        x1 = int((min(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y1 = int((min(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        x2 = int((max(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y2 = int((max(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        
        # Clamp ai limiti dell'immagine
        x1 = max(0, min(x1, img_width))
        y1 = max(0, min(y1, img_height))
        x2 = max(0, min(x2, img_width))
        y2 = max(0, min(y2, img_height))
        
        if x2 > x1 and y2 > y1:
            self.original_image = self.current_image.crop((x1, y1, x2, y2))
            self.apply_adjustments()
            self.save_to_history()  # Salva DOPO la modifica
            
        self.crop_mode = False
        self.crop_start = None
        self.crop_end = None
        self.crop_rect = None
        self.canvas.config(cursor="arrow")
        
    def apply_mirror(self):
        """Applica l'effetto specchio (flip orizzontale)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.original_image = self.current_image.transpose(Image.FLIP_LEFT_RIGHT)
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
        
    def apply_rotation(self):
        """Ruota l'immagine di 90 gradi in senso orario"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.original_image = self.current_image.rotate(-90, expand=True)
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
    
    def add_text(self):
        """Apre una finestra di dialogo per aggiungere testo all'immagine"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Crea finestra di dialogo con dimensioni maggiori
        dialog = tk.Toplevel(self.root)
        dialog.title("Aggiungi Testo")
        dialog.geometry("500x400")
        dialog.configure(bg="#2d2d2d")
        dialog.transient(self.root)
        dialog.grab_set()
        
        # Frame principale
        main_frame = ttk.Frame(dialog, style='Modern.TFrame', padding=20)
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Campo testo
        ttk.Label(main_frame, text="Testo:", style='Modern.TLabel', 
                 font=('Segoe UI', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        text_entry = tk.Text(main_frame, height=4, width=50, bg="#3d3d3d", fg="white", 
                            font=('Segoe UI', 11), insertbackground='white',
                            wrap=tk.WORD, relief=tk.FLAT, padx=10, pady=10)
        text_entry.pack(fill=tk.X, pady=(0, 20))
        text_entry.insert('1.0', 'Il tuo testo qui')
        text_entry.focus()
        text_entry.tag_configure("sel", background="#4a9eff")
        
        # Dimensione font
        ttk.Label(main_frame, text="Dimensione Font:", style='Modern.TLabel',
                 font=('Segoe UI', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        size_var = tk.IntVar(value=50)
        size_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        size_frame.pack(fill=tk.X, pady=(0, 20))
        size_slider = ttk.Scale(size_frame, from_=10, to=200, variable=size_var, 
                               orient=tk.HORIZONTAL, style='Modern.Horizontal.TScale')
        size_slider.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 10))
        size_label = ttk.Label(size_frame, text=f"{size_var.get()}", style='Modern.TLabel', 
                              width=5, font=('Segoe UI', 11, 'bold'))
        size_label.pack(side=tk.LEFT)
        
        def update_size_label(*args):
            size_label.config(text=f"{size_var.get()}")
        size_var.trace_add('write', update_size_label)
        
        # Colore
        ttk.Label(main_frame, text="Colore:", style='Modern.TLabel',
                 font=('Segoe UI', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        color_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        color_frame.pack(fill=tk.X, pady=(0, 20))
        
        color_var = tk.StringVar(value="#FFFFFF")
        selected_button = [None]  # Lista per tenere riferimento al bottone selezionato
        
        colors = [
            ("#FFFFFF", "Bianco"), ("#000000", "Nero"), ("#FF0000", "Rosso"),
            ("#00FF00", "Verde"), ("#0000FF", "Blu"), ("#FFFF00", "Giallo"),
            ("#FF00FF", "Magenta"), ("#00FFFF", "Ciano")
        ]
        
        color_buttons = []
        
        def select_color(hex_color, btn):
            color_var.set(hex_color)
            # Resetta tutti i bottoni
            for b in color_buttons:
                b.config(width=4, height=2, relief=tk.RAISED, borderwidth=2)
            # Ingrandisci il bottone selezionato
            btn.config(width=5, height=2, relief=tk.SUNKEN, borderwidth=4)
            selected_button[0] = btn
        
        for i, (hex_color, name) in enumerate(colors):
            # Crea un frame per ogni colore così posso centrarlo
            btn_container = tk.Frame(color_frame, bg="#2d2d2d")
            btn_container.pack(side=tk.LEFT, padx=3)
            
            btn = tk.Button(btn_container, bg=hex_color, width=4, height=2,
                          command=lambda c=hex_color, b=None: select_color(c, b),
                          relief=tk.RAISED, borderwidth=2, cursor="hand2")
            btn.pack()
            btn.config(command=lambda c=hex_color, b=btn: select_color(c, b))
            color_buttons.append(btn)
            
            # Aggiungi etichetta sotto il bottone per Mac
            lbl = tk.Label(btn_container, text=name.split()[0][:4], 
                          bg="#2d2d2d", fg="#999999", font=('Segoe UI', 8))
            lbl.pack()
            
            # Seleziona bianco di default
            if hex_color == "#FFFFFF":
                select_color(hex_color, btn)
        
        # Pulsanti azione
        button_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        button_frame.pack(fill=tk.X, pady=(15, 0))
        
        def on_ok():
            text_content = text_entry.get('1.0', 'end-1c').strip()
            if text_content:
                self.apply_text_to_image(text_content, size_var.get(), color_var.get())
                dialog.destroy()
            else:
                messagebox.showwarning("Attenzione", "Inserisci del testo!", parent=dialog)
        
        def on_cancel():
            dialog.destroy()
        
        ok_btn = ttk.Button(button_frame, text="Aggiungi", command=on_ok, 
                           style='Modern.TButton', width=15)
        ok_btn.pack(side=tk.LEFT, padx=5)
        
        cancel_btn = ttk.Button(button_frame, text="Annulla", command=on_cancel, 
                               style='Modern.TButton', width=15)
        cancel_btn.pack(side=tk.LEFT, padx=5)
        
        # Enter per confermare
        dialog.bind('<Return>', lambda e: on_ok())
        dialog.bind('<Escape>', lambda e: on_cancel())
    
    def apply_text_to_image(self, text, font_size, color):
        """Rende il testo mobile sul canvas per posizionamento personalizzato"""
        if self.current_image is None:
            return
        
        # Salva le informazioni del testo per il rendering finale
        text_info = {
            'text': text,
            'size': font_size,
            'color': color,
            'x': None,  # Sarà impostato quando l'utente lo posiziona
            'y': None,
            'canvas_item': None
        }
        
        # Mostra il testo sul canvas come oggetto mobile
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        # Posizione iniziale al centro
        start_x = canvas_width // 2
        start_y = canvas_height // 2
        
        # Crea il testo sul canvas con outline per visibilità
        outline_color = "black" if color == "#FFFFFF" else "white"
        
        # Crea l'outline
        outline = self.canvas.create_text(
            start_x, start_y,
            text=text,
            fill=outline_color,
            font=('Arial', font_size, 'bold'),
            tags="temp_text_outline"
        )
        
        # Crea il testo principale
        canvas_text = self.canvas.create_text(
            start_x, start_y,
            text=text,
            fill=color,
            font=('Arial', font_size),
            tags="temp_text"
        )
        
        text_info['canvas_item'] = canvas_text
        text_info['outline_item'] = outline
        
        # Variabili per il drag
        drag_data = {"x": 0, "y": 0, "item": canvas_text, "outline": outline}
        
        def start_drag(event):
            drag_data["x"] = event.x
            drag_data["y"] = event.y
        
        def do_drag(event):
            dx = event.x - drag_data["x"]
            dy = event.y - drag_data["y"]
            self.canvas.move(drag_data["item"], dx, dy)
            self.canvas.move(drag_data["outline"], dx, dy)
            drag_data["x"] = event.x
            drag_data["y"] = event.y
        
        def end_drag(event):
            # Salva la posizione finale
            coords = self.canvas.coords(drag_data["item"])
            text_info['x'] = coords[0]
            text_info['y'] = coords[1]
        
        # Bind degli eventi per rendere il testo draggabile
        self.canvas.tag_bind("temp_text", "<ButtonPress-1>", start_drag)
        self.canvas.tag_bind("temp_text", "<B1-Motion>", do_drag)
        self.canvas.tag_bind("temp_text", "<ButtonRelease-1>", end_drag)
        
        # Crea dialog per conferma
        confirm_dialog = tk.Toplevel(self.root)
        confirm_dialog.title("Posiziona Testo")
        confirm_dialog.geometry("300x150")
        confirm_dialog.configure(bg="#2d2d2d")
        confirm_dialog.transient(self.root)
        
        frame = ttk.Frame(confirm_dialog, style='Modern.TFrame', padding=20)
        frame.pack(fill=tk.BOTH, expand=True)
        
        ttk.Label(frame, 
                 text="Trascina il testo nella posizione desiderata\npoi clicca Applica",
                 style='Modern.TLabel',
                 justify=tk.CENTER).pack(pady=20)
        
        button_frame = ttk.Frame(frame, style='Modern.TFrame')
        button_frame.pack()
        
        def apply_text():
            # Ottieni coordinate finali
            coords = self.canvas.coords(canvas_text)
            
            # Converti coordinate canvas in coordinate immagine
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            
            img_w, img_h = self.current_image.size
            ratio = min(canvas_w / img_w, canvas_h / img_h) * 0.9
            
            display_w = int(img_w * ratio)
            display_h = int(img_h * ratio)
            
            offset_x = (canvas_w - display_w) // 2
            offset_y = (canvas_h - display_h) // 2
            
            # Coordinate relative all'immagine
            img_x = int((coords[0] - offset_x) / ratio)
            img_y = int((coords[1] - offset_y) / ratio)
            
            # Applica il testo all'immagine
            from PIL import ImageDraw, ImageFont
            
            img_with_text = self.original_image.copy()
            draw = ImageDraw.Draw(img_with_text)
            
            # Carica font
            try:
                font_options = [
                    'arial.ttf', 'Arial.ttf',
                    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                    '/System/Library/Fonts/Helvetica.ttc',
                    'C:\\Windows\\Fonts\\arial.ttf'
                ]
                font = None
                for font_path in font_options:
                    try:
                        font = ImageFont.truetype(font_path, font_size)
                        break
                    except:
                        continue
                
                if font is None:
                    font = ImageFont.load_default()
            except:
                font = ImageFont.load_default()
            
            # Disegna bordo
            outline_col = "#000000" if color == "#FFFFFF" else "#FFFFFF"
            for adj_x in range(-2, 3):
                for adj_y in range(-2, 3):
                    if adj_x != 0 or adj_y != 0:
                        draw.text((img_x + adj_x, img_y + adj_y), text, font=font, fill=outline_col)
            
            # Disegna testo
            draw.text((img_x, img_y), text, font=font, fill=color)
            
            self.original_image = img_with_text
            self.apply_adjustments()
            self.save_to_history()  # Salva DOPO la modifica
            
            # Pulisci il canvas
            self.canvas.delete("temp_text")
            self.canvas.delete("temp_text_outline")
            confirm_dialog.destroy()
            
            messagebox.showinfo("Successo", "Testo applicato all'immagine!")
        
        def cancel_text():
            self.canvas.delete("temp_text")
            self.canvas.delete("temp_text_outline")
            confirm_dialog.destroy()
        
        apply_btn = ttk.Button(button_frame, text="Applica", command=apply_text, style='Modern.TButton')
        apply_btn.pack(side=tk.LEFT, padx=5)
        
        cancel_btn = ttk.Button(button_frame, text="Annulla", command=cancel_text, style='Modern.TButton')
        cancel_btn.pack(side=tk.LEFT, padx=5)
        
    def reset_image(self):
        """Resetta tutte le modifiche e torna all'immagine originale"""
        if not self.history:
            return
        
        # Torna alla primissima immagine caricata
        first_state = self.history[0]
        self.original_image = first_state['image'].copy()
        
        # Reset di tutti i controlli
        self.brightness_var.set(1.0)
        self.contrast_var.set(1.0)
        self.temperature_var.set(0)
        
        # Reset di tutti i filtri
        self.vintage_active = False
        self.bw_active = False
        self.update_filter_buttons()
        
        # Resetta la cronologia mantenendo solo lo stato originale
        self.history = [first_state]
        
        self.apply_adjustments()
        
    def save_image(self, format_type):
        """Salva l'immagine nel formato specificato"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Nessuna immagine da salvare!")
            return
            
        file_extension = f".{format_type}"
        file_types = [(format_type.upper(), f"*{file_extension}")]
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=file_extension,
            filetypes=file_types,
            title=f"Salva come {format_type.upper()}"
        )
        
        if file_path:
            try:
                if format_type.lower() == 'jpg':
                    # Converti in RGB se necessario (JPG non supporta trasparenza)
                    img_to_save = self.current_image
                    if img_to_save.mode in ('RGBA', 'LA', 'P'):
                        rgb_img = Image.new('RGB', img_to_save.size, (255, 255, 255))
                        if img_to_save.mode == 'P':
                            img_to_save = img_to_save.convert('RGBA')
                        rgb_img.paste(img_to_save, mask=img_to_save.split()[-1] if img_to_save.mode == 'RGBA' else None)
                        img_to_save = rgb_img
                    img_to_save.save(file_path, 'JPEG', quality=95)
                else:
                    self.current_image.save(file_path, 'PNG')
                    
                messagebox.showinfo("Successo", f"Immagine salvata con successo in:\n{file_path}")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile salvare l'immagine:\n{str(e)}")


def main():
    """Funzione principale"""
    root = tk.Tk()
    app = PhotoEditor(root)
    
    # Configura il ridimensionamento della finestra
    root.minsize(800, 600)
    
    # Centra la finestra
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()


## Prompt 6

Undo è stato finalmente risolto.      
Ora concentriamoci su 'Testo', che presenta ancora le seguenti problematiche:      
- Quando visualizzi l'anteprima del testo da spostare, esso è sempre circa il doppio piu grande di quello che effettivamente sarà appena clicco applica.       
- Durante l'anteprima, l'outline bianco è tutto sfasato e non è centrato. Sembra piu lo stesso testo in bianco decentrato.        
- La finestra pop up deve essere 40 px piu alta e 50px piu larga, ma il contenuto al suo interno deve restare di questa grandezza.         

In [4]:
"""
Editor di Foto Professionale
Applicazione moderna per la modifica di immagini con interfaccia intuitiva
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk, ImageEnhance, ImageFilter, ImageOps
import numpy as np
from typing import Optional
import os


class PhotoEditor:
    """Editor di foto con interfaccia grafica moderna"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Editor di Foto Professionale")
        self.root.geometry("1200x800")
        self.root.configure(bg="#1a1a1a")
        
        # Variabili per le immagini
        self.original_image: Optional[Image.Image] = None
        self.current_image: Optional[Image.Image] = None
        self.display_image: Optional[ImageTk.PhotoImage] = None
        self.crop_mode = False
        self.crop_start = None
        self.crop_rect = None
        
        # Cronologia per Undo (stack di immagini)
        self.history = []
        self.max_history = 20  # Massimo 20 stati nella cronologia
        
        # Variabili per i filtri
        self.brightness_var = tk.DoubleVar(value=1.0)
        self.contrast_var = tk.DoubleVar(value=1.0)
        self.temperature_var = tk.DoubleVar(value=0)
        
        # Stati dei filtri toggle
        self.vintage_active = False
        self.bw_active = False
        
        # Riferimenti ai pulsanti per cambio colore
        self.vintage_btn = None
        self.bw_btn = None
        
        # Gestione testi sull'immagine
        self.text_items = []  # Lista di dizionari con info sui testi
        self.selected_text = None
        self.text_drag_data = {"x": 0, "y": 0}
        
        # Configurazione dello stile moderno
        self.setup_styles()
        
        # Creazione dell'interfaccia
        self.create_widgets()
        
        # Binding degli eventi e salva gli ID dei trace
        self.brightness_trace_id = self.brightness_var.trace_add('write', self.apply_adjustments)
        self.contrast_trace_id = self.contrast_var.trace_add('write', self.apply_adjustments)
        self.temperature_trace_id = self.temperature_var.trace_add('write', self.apply_adjustments)
        
    def setup_styles(self):
        """Configura gli stili moderni per l'interfaccia"""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colori moderni
        bg_dark = "#1a1a1a"
        bg_medium = "#2d2d2d"
        bg_light = "#3d3d3d"
        accent = "#4a9eff"
        text_color = "#ffffff"
        
        # Stile per i pulsanti
        style.configure('Modern.TButton',
                       background=bg_medium,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10))
        style.map('Modern.TButton',
                 background=[('active', bg_light), ('pressed', accent)])
        
        # Stile per i pulsanti attivi (filtri applicati)
        style.configure('Active.TButton',
                       background=accent,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10, 'bold'))
        style.map('Active.TButton',
                 background=[('active', '#3a8eef'), ('pressed', '#2a7edf')])
        
        # Stile per le etichette
        style.configure('Modern.TLabel',
                       background=bg_dark,
                       foreground=text_color,
                       font=('Segoe UI', 10))
        
        # Stile per i frame
        style.configure('Modern.TFrame',
                       background=bg_dark)
        
        # Stile per gli slider
        style.configure('Modern.Horizontal.TScale',
                       background=bg_dark,
                       troughcolor=bg_medium,
                       borderwidth=0,
                       sliderthickness=20)
        
    def create_widgets(self):
        """Crea tutti i widget dell'interfaccia"""
        # Frame principale
        main_frame = ttk.Frame(self.root, style='Modern.TFrame')
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Area superiore: canvas per l'immagine
        self.canvas_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        self.canvas_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 10))
        
        self.canvas = tk.Canvas(self.canvas_frame,
                               bg="#2d2d2d",
                               highlightthickness=0,
                               cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Salva il riferimento al placeholder
        self.placeholder_text = None
        
        # Crea il placeholder dopo che il canvas è stato disegnato
        self.canvas.update_idletasks()
        self.create_placeholder()
        
        # Binding per il crop
        self.canvas.bind("<Button-1>", self.on_crop_start)
        self.canvas.bind("<B1-Motion>", self.on_crop_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_end)
        
        # Binding per ridimensionamento finestra
        self.canvas.bind("<Configure>", self.on_canvas_resize)
        
        # Area inferiore: controlli
        controls_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        controls_frame.pack(fill=tk.X)
        
        # Frame per i pulsanti principali - organizzato in righe per essere responsive
        buttons_container = ttk.Frame(controls_frame, style='Modern.TFrame')
        buttons_container.pack(fill=tk.X, pady=(0, 10))
        
        # Prima riga: Caricamento e Salvataggio
        row1_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row1_frame.pack(fill=tk.X, pady=2)
        
        load_btn = ttk.Button(row1_frame,
                             text="📁 Carica Immagine",
                             command=self.load_image,
                             style='Modern.TButton')
        load_btn.pack(side=tk.LEFT, padx=5)
        
        save_jpg_btn = ttk.Button(row1_frame,
                                 text="💾 Salva JPG",
                                 command=lambda: self.save_image('jpg'),
                                 style='Modern.TButton')
        save_jpg_btn.pack(side=tk.LEFT, padx=5)
        
        save_png_btn = ttk.Button(row1_frame,
                                 text="💾 Salva PNG",
                                 command=lambda: self.save_image('png'),
                                 style='Modern.TButton')
        save_png_btn.pack(side=tk.LEFT, padx=5)
        
        # Seconda riga: Filtri
        row2_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row2_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row2_frame, text="Filtri:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        self.vintage_btn = ttk.Button(row2_frame,
                                text="🎨 Vintage",
                                command=self.toggle_vintage,
                                style='Modern.TButton')
        self.vintage_btn.pack(side=tk.LEFT, padx=5)
        
        self.bw_btn = ttk.Button(row2_frame,
                           text="⚫ Bianco e Nero",
                           command=self.toggle_bw,
                           style='Modern.TButton')
        self.bw_btn.pack(side=tk.LEFT, padx=5)
        
        # Terza riga: Trasformazioni
        row3_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row3_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row3_frame, text="Trasformazioni:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        crop_btn = ttk.Button(row3_frame,
                             text="✂️ Ritaglia",
                             command=self.toggle_crop_mode,
                             style='Modern.TButton')
        crop_btn.pack(side=tk.LEFT, padx=5)
        
        mirror_btn = ttk.Button(row3_frame,
                               text="🔄 Specchio",
                               command=self.apply_mirror,
                               style='Modern.TButton')
        mirror_btn.pack(side=tk.LEFT, padx=5)
        
        rotate_btn = ttk.Button(row3_frame,
                               text="🔃 Ruota 90°",
                               command=self.apply_rotation,
                               style='Modern.TButton')
        rotate_btn.pack(side=tk.LEFT, padx=5)
        
        text_btn = ttk.Button(row3_frame,
                             text="📝 Testo",
                             command=self.add_text,
                             style='Modern.TButton')
        text_btn.pack(side=tk.LEFT, padx=5)
        
        # Quarta riga: Undo e Reset
        row4_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row4_frame.pack(fill=tk.X, pady=2)
        
        ttk.Label(row4_frame, text="Controlli:", style='Modern.TLabel').pack(side=tk.LEFT, padx=5)
        
        undo_btn = ttk.Button(row4_frame,
                             text="⮌ Undo",
                             command=self.undo,
                             style='Modern.TButton')
        undo_btn.pack(side=tk.LEFT, padx=5)
        
        reset_btn = ttk.Button(row4_frame,
                              text="↺ Reset",
                              command=self.reset_image,
                              style='Modern.TButton')
        reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Frame per gli slider
        sliders_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        sliders_frame.pack(fill=tk.X)
        
        # Slider luminosità
        self.create_slider(sliders_frame, "☀️ Luminosità", self.brightness_var, 0.0, 2.0, 0)
        
        # Slider contrasto
        self.create_slider(sliders_frame, "◐ Contrasto", self.contrast_var, 0.0, 2.0, 1)
        
        # Slider temperatura
        self.create_slider(sliders_frame, "🌡️ Temperatura", self.temperature_var, -100, 100, 2)
        
    def create_slider(self, parent, label_text, variable, from_, to, column):
        """Crea uno slider con etichetta"""
        frame = ttk.Frame(parent, style='Modern.TFrame')
        frame.grid(row=0, column=column, padx=10, pady=5, sticky='ew')
        parent.columnconfigure(column, weight=1)
        
        label = ttk.Label(frame, text=label_text, style='Modern.TLabel')
        label.pack(anchor='w')
        
        slider = ttk.Scale(frame,
                          from_=from_,
                          to=to,
                          variable=variable,
                          orient=tk.HORIZONTAL,
                          style='Modern.Horizontal.TScale')
        slider.pack(fill=tk.X, pady=5)
        
        # Etichetta valore
        value_label = ttk.Label(frame, text=f"{variable.get():.2f}", style='Modern.TLabel')
        value_label.pack(anchor='e')
        
        def update_label(*args):
            value_label.config(text=f"{variable.get():.2f}")
        
        variable.trace_add('write', update_label)
        
    def load_image(self):
        """Carica un'immagine dal file system"""
        file_path = filedialog.askopenfilename(
            title="Seleziona un'immagine",
            filetypes=[
                ("Immagini", "*.jpg *.jpeg *.png *.bmp *.gif"),
                ("JPEG", "*.jpg *.jpeg"),
                ("PNG", "*.png"),
                ("Tutti i file", "*.*")
            ]
        )
        
        if file_path:
            try:
                self.original_image = Image.open(file_path)
                self.current_image = self.original_image.copy()
                
                # Reset di tutti i controlli e filtri
                self.brightness_var.set(1.0)
                self.contrast_var.set(1.0)
                self.temperature_var.set(0)
                
                # Reset stato filtri
                self.vintage_active = False
                self.bw_active = False
                self.update_filter_buttons()
                
                # Reset cronologia
                self.history = []
                self.save_to_history()
                
                self.display_current_image()
                self.canvas.delete("placeholder")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile caricare l'immagine:\n{str(e)}")
                
    def display_current_image(self):
        """Visualizza l'immagine corrente sul canvas"""
        if self.current_image is None:
            return
            
        # Calcola le dimensioni per il fit
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
            
        # Calcola il ridimensionamento mantenendo l'aspect ratio
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        new_width = int(img_width * ratio * 0.9)  # 90% per margini
        new_height = int(img_height * ratio * 0.9)
        
        # Ridimensiona l'immagine
        display_img = self.current_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Converti in PhotoImage
        self.display_image = ImageTk.PhotoImage(display_img)
        
        # Pulisci il canvas e mostra l'immagine
        self.canvas.delete("all")
        self.canvas.create_image(
            canvas_width // 2,
            canvas_height // 2,
            image=self.display_image,
            anchor=tk.CENTER,
            tags="image"
        )
        
    def save_to_history(self):
        """Salva lo stato corrente nella cronologia"""
        if self.original_image is None:
            return
            
        # Salva una copia dell'immagine e dello stato dei filtri
        state = {
            'image': self.original_image.copy(),
            'brightness': self.brightness_var.get(),
            'contrast': self.contrast_var.get(),
            'temperature': self.temperature_var.get(),
            'vintage': self.vintage_active,
            'bw': self.bw_active
        }
        
        self.history.append(state)
        
        # Limita la dimensione della cronologia
        if len(self.history) > self.max_history:
            self.history.pop(0)
            
    def undo(self):
        """Annulla l'ultima modifica"""
        if len(self.history) <= 1:
            messagebox.showinfo("Undo", "Nessuna operazione da annullare!")
            return
            
        # Rimuovi lo stato corrente e torna al precedente
        self.history.pop()
        
        # Ripristina lo stato precedente
        if self.history:
            previous_state = self.history[-1]
            self.original_image = previous_state['image'].copy()
            
            # Disattiva temporaneamente i trace per evitare salvataggi in cronologia
            self.brightness_var.trace_remove('write', self.brightness_trace_id)
            self.contrast_var.trace_remove('write', self.contrast_trace_id)
            self.temperature_var.trace_remove('write', self.temperature_trace_id)
            
            self.brightness_var.set(previous_state['brightness'])
            self.contrast_var.set(previous_state['contrast'])
            self.temperature_var.set(previous_state['temperature'])
            
            # Riattiva i trace
            self.brightness_trace_id = self.brightness_var.trace_add('write', self.apply_adjustments)
            self.contrast_trace_id = self.contrast_var.trace_add('write', self.apply_adjustments)
            self.temperature_trace_id = self.temperature_var.trace_add('write', self.apply_adjustments)
            
            self.vintage_active = previous_state['vintage']
            self.bw_active = previous_state['bw']
            
            self.update_filter_buttons()
            self.apply_adjustments()
            
    def update_filter_buttons(self):
        """Aggiorna lo stile dei pulsanti filtro in base allo stato"""
        if self.vintage_btn:
            style = 'Active.TButton' if self.vintage_active else 'Modern.TButton'
            self.vintage_btn.configure(style=style)
            
        if self.bw_btn:
            style = 'Active.TButton' if self.bw_active else 'Modern.TButton'
            self.bw_btn.configure(style=style)
    
    def create_placeholder(self):
        """Crea il testo placeholder centrato nel canvas"""
        if self.placeholder_text:
            self.canvas.delete(self.placeholder_text)
        
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        # Usa dimensioni di default se il canvas non è ancora renderizzato
        if canvas_width <= 1:
            canvas_width = 800
        if canvas_height <= 1:
            canvas_height = 600
        
        self.placeholder_text = self.canvas.create_text(
            canvas_width // 2,
            canvas_height // 2,
            text="Carica un'immagine per iniziare",
            fill="#666666",
            font=('Segoe UI', 16),
            tags="placeholder"
        )
    
    def on_canvas_resize(self, event):
        """Gestisce il ridimensionamento del canvas"""
        if self.original_image is None and self.placeholder_text:
            # Riposiziona il placeholder al centro
            self.create_placeholder()
        elif self.current_image is not None:
            # Ridisegna l'immagine per adattarla alle nuove dimensioni
            self.display_current_image()
    
    def get_base_image(self):
        """Ottiene l'immagine base senza filtri cromatici dalla cronologia"""
        # Cerca l'ultima immagine senza filtri vintage o b/n
        for i in range(len(self.history) - 1, -1, -1):
            state = self.history[i]
            if not state['vintage'] and not state['bw']:
                return state['image'].copy()
        
        # Se non trovata, usa la prima immagine della cronologia
        if self.history:
            return self.history[0]['image'].copy()
        
        # Fallback (non dovrebbe mai succedere)
        return self.original_image.copy()
        
    def apply_adjustments(self, *args):
        """Applica le regolazioni di luminosità, contrasto e temperatura"""
        if self.original_image is None:
            return
            
        # Parte dall'immagine originale
        img = self.original_image.copy()
        
        # Applica luminosità
        if self.brightness_var.get() != 1.0:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(self.brightness_var.get())
            
        # Applica contrasto
        if self.contrast_var.get() != 1.0:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(self.contrast_var.get())
            
        # Applica temperatura (modifica del bilanciamento del colore)
        if self.temperature_var.get() != 0:
            img = self.adjust_temperature(img, self.temperature_var.get())
            
        self.current_image = img
        self.display_current_image()
        
    def adjust_temperature(self, image, value):
        """Regola la temperatura del colore dell'immagine"""
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        # Converti in array numpy
        img_array = np.array(image, dtype=np.float32)
        
        # Applica shift di temperatura
        if value > 0:  # Più caldo (più rosso/giallo)
            img_array[:, :, 0] += value * 0.5  # Rosso
            img_array[:, :, 1] += value * 0.3  # Verde
        else:  # Più freddo (più blu)
            img_array[:, :, 2] += abs(value) * 0.5  # Blu
            
        # Clamp values
        img_array = np.clip(img_array, 0, 255)
        
        return Image.fromarray(img_array.astype(np.uint8))
        
    def toggle_vintage(self):
        """Attiva/disattiva il filtro vintage (seppia)"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Toggle dello stato
        self.vintage_active = not self.vintage_active
        
        # Se stiamo attivando vintage, disattiva B/N
        if self.vintage_active and self.bw_active:
            self.bw_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.vintage_active:
            # Applica filtro vintage sull'immagine base
            img = base_image.convert('RGB')
            img_array = np.array(img, dtype=np.float32)
            
            # Matrice seppia
            sepia_filter = np.array([
                [0.393, 0.769, 0.189],
                [0.349, 0.686, 0.168],
                [0.272, 0.534, 0.131]
            ])
            
            # Applica il filtro
            sepia_img = img_array @ sepia_filter.T
            sepia_img = np.clip(sepia_img, 0, 255)
            
            self.original_image = Image.fromarray(sepia_img.astype(np.uint8))
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
        
    def toggle_bw(self):
        """Attiva/disattiva il filtro bianco e nero"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Toggle dello stato
        self.bw_active = not self.bw_active
        
        # Se stiamo attivando B/N, disattiva Vintage
        if self.bw_active and self.vintage_active:
            self.vintage_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.bw_active:
            # Applica filtro bianco e nero sull'immagine base
            self.original_image = base_image.convert('L').convert('RGB')
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
        
    def toggle_crop_mode(self):
        """Attiva/disattiva la modalità ritaglio"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.crop_mode = not self.crop_mode
        if self.crop_mode:
            self.canvas.config(cursor="cross")
            messagebox.showinfo("Modalità Ritaglio",
                              "Trascina il mouse sull'immagine per selezionare l'area da ritagliare.\n"
                              "Clicca di nuovo su 'Ritaglia' per applicare.")
        else:
            self.canvas.config(cursor="arrow")
            if self.crop_rect:
                self.apply_crop()
                
    def on_crop_start(self, event):
        """Inizia la selezione per il ritaglio"""
        if not self.crop_mode:
            return
        self.crop_start = (event.x, event.y)
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
    def on_crop_drag(self, event):
        """Disegna il rettangolo di selezione"""
        if not self.crop_mode or not self.crop_start:
            return
            
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
        self.crop_rect = self.canvas.create_rectangle(
            self.crop_start[0], self.crop_start[1],
            event.x, event.y,
            outline='#4a9eff',
            width=2,
            dash=(5, 5)
        )
        
    def on_crop_end(self, event):
        """Finalizza la selezione"""
        if not self.crop_mode or not self.crop_start:
            return
        self.crop_end = (event.x, event.y)
        
    def apply_crop(self):
        """Applica il ritaglio all'immagine"""
        if not self.crop_start or not self.crop_end:
            return
            
        # Calcola le coordinate relative all'immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Converti coordinate canvas in coordinate immagine
        x1 = int((min(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y1 = int((min(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        x2 = int((max(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y2 = int((max(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        
        # Clamp ai limiti dell'immagine
        x1 = max(0, min(x1, img_width))
        y1 = max(0, min(y1, img_height))
        x2 = max(0, min(x2, img_width))
        y2 = max(0, min(y2, img_height))
        
        if x2 > x1 and y2 > y1:
            self.original_image = self.current_image.crop((x1, y1, x2, y2))
            self.apply_adjustments()
            self.save_to_history()  # Salva DOPO la modifica
            
        self.crop_mode = False
        self.crop_start = None
        self.crop_end = None
        self.crop_rect = None
        self.canvas.config(cursor="arrow")
        
    def apply_mirror(self):
        """Applica l'effetto specchio (flip orizzontale)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.original_image = self.current_image.transpose(Image.FLIP_LEFT_RIGHT)
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
        
    def apply_rotation(self):
        """Ruota l'immagine di 90 gradi in senso orario"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.original_image = self.current_image.rotate(-90, expand=True)
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
    
    def add_text(self):
        """Apre una finestra di dialogo per aggiungere testo all'immagine"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Crea finestra di dialogo con dimensioni maggiori
        dialog = tk.Toplevel(self.root)
        dialog.title("Aggiungi Testo")
        dialog.geometry("550x440")  # +50px larghezza, +40px altezza
        dialog.configure(bg="#2d2d2d")
        dialog.transient(self.root)
        dialog.grab_set()
        
        # Frame principale
        main_frame = ttk.Frame(dialog, style='Modern.TFrame', padding=20)
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Campo testo
        ttk.Label(main_frame, text="Testo:", style='Modern.TLabel', 
                 font=('Segoe UI', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        text_entry = tk.Text(main_frame, height=4, width=50, bg="#3d3d3d", fg="white", 
                            font=('Segoe UI', 11), insertbackground='white',
                            wrap=tk.WORD, relief=tk.FLAT, padx=10, pady=10)
        text_entry.pack(fill=tk.X, pady=(0, 20))
        text_entry.insert('1.0', 'Il tuo testo qui')
        text_entry.focus()
        text_entry.tag_configure("sel", background="#4a9eff")
        
        # Dimensione font
        ttk.Label(main_frame, text="Dimensione Font:", style='Modern.TLabel',
                 font=('Segoe UI', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        size_var = tk.IntVar(value=50)
        size_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        size_frame.pack(fill=tk.X, pady=(0, 20))
        size_slider = ttk.Scale(size_frame, from_=10, to=200, variable=size_var, 
                               orient=tk.HORIZONTAL, style='Modern.Horizontal.TScale')
        size_slider.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 10))
        size_label = ttk.Label(size_frame, text=f"{size_var.get()}", style='Modern.TLabel', 
                              width=5, font=('Segoe UI', 11, 'bold'))
        size_label.pack(side=tk.LEFT)
        
        def update_size_label(*args):
            size_label.config(text=f"{size_var.get()}")
        size_var.trace_add('write', update_size_label)
        
        # Colore
        ttk.Label(main_frame, text="Colore:", style='Modern.TLabel',
                 font=('Segoe UI', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        color_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        color_frame.pack(fill=tk.X, pady=(0, 20))
        
        color_var = tk.StringVar(value="#FFFFFF")
        selected_button = [None]  # Lista per tenere riferimento al bottone selezionato
        
        colors = [
            ("#FFFFFF", "Bianco"), ("#000000", "Nero"), ("#FF0000", "Rosso"),
            ("#00FF00", "Verde"), ("#0000FF", "Blu"), ("#FFFF00", "Giallo"),
            ("#FF00FF", "Magenta"), ("#00FFFF", "Ciano")
        ]
        
        color_buttons = []
        
        def select_color(hex_color, btn):
            color_var.set(hex_color)
            # Resetta tutti i bottoni
            for b in color_buttons:
                b.config(width=4, height=2, relief=tk.RAISED, borderwidth=2)
            # Ingrandisci il bottone selezionato
            btn.config(width=5, height=2, relief=tk.SUNKEN, borderwidth=4)
            selected_button[0] = btn
        
        for i, (hex_color, name) in enumerate(colors):
            # Crea un frame per ogni colore così posso centrarlo
            btn_container = tk.Frame(color_frame, bg="#2d2d2d")
            btn_container.pack(side=tk.LEFT, padx=3)
            
            btn = tk.Button(btn_container, bg=hex_color, width=4, height=2,
                          command=lambda c=hex_color, b=None: select_color(c, b),
                          relief=tk.RAISED, borderwidth=2, cursor="hand2")
            btn.pack()
            btn.config(command=lambda c=hex_color, b=btn: select_color(c, b))
            color_buttons.append(btn)
            
            # Aggiungi etichetta sotto il bottone per Mac
            lbl = tk.Label(btn_container, text=name.split()[0][:4], 
                          bg="#2d2d2d", fg="#999999", font=('Segoe UI', 8))
            lbl.pack()
            
            # Seleziona bianco di default
            if hex_color == "#FFFFFF":
                select_color(hex_color, btn)
        
        # Pulsanti azione
        button_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        button_frame.pack(fill=tk.X, pady=(15, 0))
        
        def on_ok():
            text_content = text_entry.get('1.0', 'end-1c').strip()
            if text_content:
                self.apply_text_to_image(text_content, size_var.get(), color_var.get())
                dialog.destroy()
            else:
                messagebox.showwarning("Attenzione", "Inserisci del testo!", parent=dialog)
        
        def on_cancel():
            dialog.destroy()
        
        ok_btn = ttk.Button(button_frame, text="Aggiungi", command=on_ok, 
                           style='Modern.TButton', width=15)
        ok_btn.pack(side=tk.LEFT, padx=5)
        
        cancel_btn = ttk.Button(button_frame, text="Annulla", command=on_cancel, 
                               style='Modern.TButton', width=15)
        cancel_btn.pack(side=tk.LEFT, padx=5)
        
        # Enter per confermare
        dialog.bind('<Return>', lambda e: on_ok())
        dialog.bind('<Escape>', lambda e: on_cancel())
    
    def apply_text_to_image(self, text, font_size, color):
        """Rende il testo mobile sul canvas per posizionamento personalizzato"""
        if self.current_image is None:
            return
        
        # Calcola il rapporto di scala tra canvas e immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Scala la dimensione del font per il canvas
        canvas_font_size = int(font_size * ratio)
        
        # Posizione iniziale al centro dell'immagine visualizzata
        start_x = canvas_width // 2
        start_y = canvas_height // 2
        
        # Crea il testo sul canvas con outline VERO (usando più testi sovrapposti)
        outline_color = "black" if color == "#FFFFFF" else "white"
        
        # Crea gli outline sovrapposti per un effetto bordo vero
        outline_items = []
        offsets = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
        
        for dx, dy in offsets:
            outline = self.canvas.create_text(
                start_x + dx, start_y + dy,
                text=text,
                fill=outline_color,
                font=('Arial', canvas_font_size, 'bold'),
                tags="temp_text_outline"
            )
            outline_items.append(outline)
        
        # Crea il testo principale sopra gli outline
        canvas_text = self.canvas.create_text(
            start_x, start_y,
            text=text,
            fill=color,
            font=('Arial', canvas_font_size, 'bold'),
            tags="temp_text"
        )
        
        # Variabili per il drag
        drag_data = {
            "x": 0, 
            "y": 0, 
            "item": canvas_text, 
            "outlines": outline_items
        }
        
        def start_drag(event):
            drag_data["x"] = event.x
            drag_data["y"] = event.y
        
        def do_drag(event):
            dx = event.x - drag_data["x"]
            dy = event.y - drag_data["y"]
            self.canvas.move(drag_data["item"], dx, dy)
            for outline in drag_data["outlines"]:
                self.canvas.move(outline, dx, dy)
            drag_data["x"] = event.x
            drag_data["y"] = event.y
        
        def end_drag(event):
            pass  # Non serve salvare qui, lo faremo al momento dell'applicazione
        
        # Bind degli eventi per rendere il testo draggabile
        self.canvas.tag_bind("temp_text", "<ButtonPress-1>", start_drag)
        self.canvas.tag_bind("temp_text", "<B1-Motion>", do_drag)
        self.canvas.tag_bind("temp_text", "<ButtonRelease-1>", end_drag)
        
        # Crea dialog per conferma
        confirm_dialog = tk.Toplevel(self.root)
        confirm_dialog.title("Posiziona Testo")
        confirm_dialog.geometry("350x150")
        confirm_dialog.configure(bg="#2d2d2d")
        confirm_dialog.transient(self.root)
        
        frame = ttk.Frame(confirm_dialog, style='Modern.TFrame', padding=20)
        frame.pack(fill=tk.BOTH, expand=True)
        
        ttk.Label(frame, 
                 text="Trascina il testo nella posizione desiderata\npoi clicca Applica",
                 style='Modern.TLabel',
                 justify=tk.CENTER).pack(pady=20)
        
        button_frame = ttk.Frame(frame, style='Modern.TFrame')
        button_frame.pack()
        
        def apply_text():
            # Ottieni coordinate finali del testo sul canvas
            coords = self.canvas.coords(canvas_text)
            
            # Converti coordinate canvas in coordinate immagine
            img_x = int((coords[0] - offset_x) / ratio)
            img_y = int((coords[1] - offset_y) / ratio)
            
            # Applica il testo all'immagine
            from PIL import ImageDraw, ImageFont
            
            img_with_text = self.original_image.copy()
            draw = ImageDraw.Draw(img_with_text)
            
            # Carica font con la dimensione ORIGINALE (non scalata)
            try:
                font_options = [
                    'arial.ttf', 'Arial.ttf',
                    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                    '/System/Library/Fonts/Helvetica.ttc',
                    'C:\\Windows\\Fonts\\arial.ttf'
                ]
                font = None
                for font_path in font_options:
                    try:
                        font = ImageFont.truetype(font_path, font_size)  # Usa font_size originale
                        break
                    except:
                        continue
                
                if font is None:
                    font = ImageFont.load_default()
            except:
                font = ImageFont.load_default()
            
            # Disegna bordo (outline) - bordo più sottile e preciso
            outline_col = "#000000" if color == "#FFFFFF" else "#FFFFFF"
            for adj_x in range(-1, 2):
                for adj_y in range(-1, 2):
                    if adj_x != 0 or adj_y != 0:
                        draw.text((img_x + adj_x, img_y + adj_y), text, font=font, fill=outline_col)
            
            # Disegna testo principale
            draw.text((img_x, img_y), text, font=font, fill=color)
            
            self.original_image = img_with_text
            self.apply_adjustments()
            self.save_to_history()  # Salva DOPO la modifica
            
            # Pulisci il canvas
            self.canvas.delete("temp_text")
            self.canvas.delete("temp_text_outline")
            confirm_dialog.destroy()
            
            messagebox.showinfo("Successo", "Testo applicato all'immagine!")
        
        def cancel_text():
            self.canvas.delete("temp_text")
            self.canvas.delete("temp_text_outline")
            confirm_dialog.destroy()
        
        apply_btn = ttk.Button(button_frame, text="Applica", command=apply_text, style='Modern.TButton')
        apply_btn.pack(side=tk.LEFT, padx=5)
        
        cancel_btn = ttk.Button(button_frame, text="Annulla", command=cancel_text, style='Modern.TButton')
        cancel_btn.pack(side=tk.LEFT, padx=5)
        
    def reset_image(self):
        """Resetta tutte le modifiche e torna all'immagine originale"""
        if not self.history:
            return
        
        # Torna alla primissima immagine caricata
        first_state = self.history[0]
        self.original_image = first_state['image'].copy()
        
        # Reset di tutti i controlli
        self.brightness_var.set(1.0)
        self.contrast_var.set(1.0)
        self.temperature_var.set(0)
        
        # Reset di tutti i filtri
        self.vintage_active = False
        self.bw_active = False
        self.update_filter_buttons()
        
        # Resetta la cronologia mantenendo solo lo stato originale
        self.history = [first_state]
        
        self.apply_adjustments()
        
    def save_image(self, format_type):
        """Salva l'immagine nel formato specificato"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Nessuna immagine da salvare!")
            return
            
        file_extension = f".{format_type}"
        file_types = [(format_type.upper(), f"*{file_extension}")]
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=file_extension,
            filetypes=file_types,
            title=f"Salva come {format_type.upper()}"
        )
        
        if file_path:
            try:
                if format_type.lower() == 'jpg':
                    # Converti in RGB se necessario (JPG non supporta trasparenza)
                    img_to_save = self.current_image
                    if img_to_save.mode in ('RGBA', 'LA', 'P'):
                        rgb_img = Image.new('RGB', img_to_save.size, (255, 255, 255))
                        if img_to_save.mode == 'P':
                            img_to_save = img_to_save.convert('RGBA')
                        rgb_img.paste(img_to_save, mask=img_to_save.split()[-1] if img_to_save.mode == 'RGBA' else None)
                        img_to_save = rgb_img
                    img_to_save.save(file_path, 'JPEG', quality=95)
                else:
                    self.current_image.save(file_path, 'PNG')
                    
                messagebox.showinfo("Successo", f"Immagine salvata con successo in:\n{file_path}")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile salvare l'immagine:\n{str(e)}")


def main():
    """Funzione principale"""
    root = tk.Tk()
    app = PhotoEditor(root)
    
    # Configura il ridimensionamento della finestra
    root.minsize(800, 600)
    
    # Centra la finestra
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()


## Prompt 7

L'outline e il pop up sono risolti.         
Il testo nell'anteprima però è ancora piu grande poi di quanto lo sarà effettivamente, e questo quindi si traduce anche in un lieve spostamento relativo del testo.         
Risolvi questa cosa e poi abbellisci l'interfaccia a tuo piacimento.      

In [5]:
"""
Editor di Foto Professionale
Applicazione moderna per la modifica di immagini con interfaccia intuitiva
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk, ImageEnhance, ImageFilter, ImageOps
import numpy as np
from typing import Optional
import os


class PhotoEditor:
    """Editor di foto con interfaccia grafica moderna"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Editor di Foto Professionale")
        self.root.geometry("1200x800")
        self.root.configure(bg="#1a1a1a")
        
        # Variabili per le immagini
        self.original_image: Optional[Image.Image] = None
        self.current_image: Optional[Image.Image] = None
        self.display_image: Optional[ImageTk.PhotoImage] = None
        self.crop_mode = False
        self.crop_start = None
        self.crop_rect = None
        
        # Cronologia per Undo (stack di immagini)
        self.history = []
        self.max_history = 20  # Massimo 20 stati nella cronologia
        
        # Variabili per i filtri
        self.brightness_var = tk.DoubleVar(value=1.0)
        self.contrast_var = tk.DoubleVar(value=1.0)
        self.temperature_var = tk.DoubleVar(value=0)
        
        # Stati dei filtri toggle
        self.vintage_active = False
        self.bw_active = False
        
        # Riferimenti ai pulsanti per cambio colore
        self.vintage_btn = None
        self.bw_btn = None
        
        # Gestione testi sull'immagine
        self.text_items = []  # Lista di dizionari con info sui testi
        self.selected_text = None
        self.text_drag_data = {"x": 0, "y": 0}
        
        # Configurazione dello stile moderno
        self.setup_styles()
        
        # Creazione dell'interfaccia
        self.create_widgets()
        
        # Binding degli eventi e salva gli ID dei trace
        self.brightness_trace_id = self.brightness_var.trace_add('write', self.apply_adjustments)
        self.contrast_trace_id = self.contrast_var.trace_add('write', self.apply_adjustments)
        self.temperature_trace_id = self.temperature_var.trace_add('write', self.apply_adjustments)
        
    def setup_styles(self):
        """Configura gli stili moderni per l'interfaccia"""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colori moderni con palette professionale
        bg_dark = "#1a1a1a"          # Nero profondo
        bg_medium = "#2d2d2d"         # Grigio scuro
        bg_light = "#3d3d3d"          # Grigio medio
        accent = "#4a9eff"            # Blu moderno
        accent_hover = "#5ba3ff"      # Blu più chiaro per hover
        accent_pressed = "#3a8eef"    # Blu più scuro per pressed
        text_color = "#ffffff"        # Bianco puro
        text_secondary = "#b0b0b0"    # Grigio chiaro per testo secondario
        success = "#4ade80"           # Verde per successo
        
        # Stile per i pulsanti normali con effetto moderno
        style.configure('Modern.TButton',
                       background=bg_medium,
                       foreground=text_color,
                       borderwidth=1,
                       relief='flat',
                       focuscolor='none',
                       padding=12,
                       font=('Segoe UI', 10, 'normal'))
        style.map('Modern.TButton',
                 background=[('active', bg_light), ('pressed', accent_pressed)],
                 foreground=[('active', text_color)],
                 relief=[('pressed', 'sunken')])
        
        # Stile per i pulsanti attivi (filtri applicati) con effetto luminoso
        style.configure('Active.TButton',
                       background=accent,
                       foreground=text_color,
                       borderwidth=0,
                       relief='flat',
                       focuscolor='none',
                       padding=12,
                       font=('Segoe UI', 10, 'bold'))
        style.map('Active.TButton',
                 background=[('active', accent_hover), ('pressed', accent_pressed)],
                 foreground=[('active', text_color)],
                 relief=[('pressed', 'sunken')])
        
        # Stile per le etichette con font migliorato
        style.configure('Modern.TLabel',
                       background=bg_dark,
                       foreground=text_color,
                       font=('Segoe UI', 10))
        
        # Stile per le etichette delle sezioni (più grandi e bold)
        style.configure('Section.TLabel',
                       background=bg_dark,
                       foreground=accent,
                       font=('Segoe UI', 11, 'bold'))
        
        # Stile per i frame
        style.configure('Modern.TFrame',
                       background=bg_dark)
        
        # Stile per gli slider con colori moderni
        style.configure('Modern.Horizontal.TScale',
                       background=bg_dark,
                       troughcolor=bg_medium,
                       borderwidth=0,
                       sliderthickness=22)
        style.map('Modern.Horizontal.TScale',
                 background=[('active', accent)],
                 troughcolor=[('active', bg_light)])
        
    def create_widgets(self):
        """Crea tutti i widget dell'interfaccia"""
        # Frame principale
        main_frame = ttk.Frame(self.root, style='Modern.TFrame')
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Area superiore: canvas per l'immagine con bordo elegante
        self.canvas_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        self.canvas_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 15))
        
        # Frame interno per creare effetto bordo
        canvas_border = tk.Frame(self.canvas_frame, bg="#4a9eff", padx=1, pady=1)
        canvas_border.pack(fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(canvas_border,
                               bg="#2d2d2d",
                               highlightthickness=0,
                               cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Salva il riferimento al placeholder
        self.placeholder_text = None
        
        # Crea il placeholder dopo che il canvas è stato disegnato
        self.canvas.update_idletasks()
        self.create_placeholder()
        
        # Binding per il crop
        self.canvas.bind("<Button-1>", self.on_crop_start)
        self.canvas.bind("<B1-Motion>", self.on_crop_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_end)
        
        # Binding per ridimensionamento finestra
        self.canvas.bind("<Configure>", self.on_canvas_resize)
        
        # Area inferiore: controlli
        controls_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        controls_frame.pack(fill=tk.X)
        
        # Frame per i pulsanti principali - organizzato in righe per essere responsive
        buttons_container = ttk.Frame(controls_frame, style='Modern.TFrame')
        buttons_container.pack(fill=tk.X, pady=(0, 10))
        
        # Prima riga: Caricamento e Salvataggio
        row1_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row1_frame.pack(fill=tk.X, pady=(5, 8))
        
        load_btn = ttk.Button(row1_frame,
                             text="📁 Carica Immagine",
                             command=self.load_image,
                             style='Modern.TButton')
        load_btn.pack(side=tk.LEFT, padx=5)
        
        save_jpg_btn = ttk.Button(row1_frame,
                                 text="💾 Salva JPG",
                                 command=lambda: self.save_image('jpg'),
                                 style='Modern.TButton')
        save_jpg_btn.pack(side=tk.LEFT, padx=5)
        
        save_png_btn = ttk.Button(row1_frame,
                                 text="💾 Salva PNG",
                                 command=lambda: self.save_image('png'),
                                 style='Modern.TButton')
        save_png_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore elegante
        separator1 = tk.Frame(buttons_container, height=1, bg="#4a9eff")
        separator1.pack(fill=tk.X, pady=5)
        
        # Seconda riga: Filtri
        row2_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row2_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(row2_frame, text="🎨 Filtri:", style='Section.TLabel').pack(side=tk.LEFT, padx=5)
        
        self.vintage_btn = ttk.Button(row2_frame,
                                text="🎨 Vintage",
                                command=self.toggle_vintage,
                                style='Modern.TButton')
        self.vintage_btn.pack(side=tk.LEFT, padx=5)
        
        self.bw_btn = ttk.Button(row2_frame,
                           text="⚫ Bianco e Nero",
                           command=self.toggle_bw,
                           style='Modern.TButton')
        self.bw_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore elegante
        separator2 = tk.Frame(buttons_container, height=1, bg="#4a9eff")
        separator2.pack(fill=tk.X, pady=5)
        
        # Terza riga: Trasformazioni
        row3_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row3_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(row3_frame, text="✨ Trasformazioni:", style='Section.TLabel').pack(side=tk.LEFT, padx=5)
        
        crop_btn = ttk.Button(row3_frame,
                             text="✂️ Ritaglia",
                             command=self.toggle_crop_mode,
                             style='Modern.TButton')
        crop_btn.pack(side=tk.LEFT, padx=5)
        
        mirror_btn = ttk.Button(row3_frame,
                               text="🔄 Specchio",
                               command=self.apply_mirror,
                               style='Modern.TButton')
        mirror_btn.pack(side=tk.LEFT, padx=5)
        
        rotate_btn = ttk.Button(row3_frame,
                               text="🔃 Ruota 90°",
                               command=self.apply_rotation,
                               style='Modern.TButton')
        rotate_btn.pack(side=tk.LEFT, padx=5)
        
        text_btn = ttk.Button(row3_frame,
                             text="📝 Testo",
                             command=self.add_text,
                             style='Modern.TButton')
        text_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore elegante
        separator3 = tk.Frame(buttons_container, height=1, bg="#4a9eff")
        separator3.pack(fill=tk.X, pady=5)
        
        # Quarta riga: Undo e Reset
        row4_frame = ttk.Frame(buttons_container, style='Modern.TFrame')
        row4_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(row4_frame, text="⚙️ Controlli:", style='Section.TLabel').pack(side=tk.LEFT, padx=5)
        
        undo_btn = ttk.Button(row4_frame,
                             text="⮌ Undo",
                             command=self.undo,
                             style='Modern.TButton')
        undo_btn.pack(side=tk.LEFT, padx=5)
        
        reset_btn = ttk.Button(row4_frame,
                              text="↺ Reset",
                              command=self.reset_image,
                              style='Modern.TButton')
        reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Frame per gli slider
        sliders_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        sliders_frame.pack(fill=tk.X)
        
        # Slider luminosità
        self.create_slider(sliders_frame, "☀️ Luminosità", self.brightness_var, 0.0, 2.0, 0)
        
        # Slider contrasto
        self.create_slider(sliders_frame, "◐ Contrasto", self.contrast_var, 0.0, 2.0, 1)
        
        # Slider temperatura
        self.create_slider(sliders_frame, "🌡️ Temperatura", self.temperature_var, -100, 100, 2)
        
    def create_slider(self, parent, label_text, variable, from_, to, column):
        """Crea uno slider con etichetta"""
        frame = ttk.Frame(parent, style='Modern.TFrame')
        frame.grid(row=0, column=column, padx=10, pady=5, sticky='ew')
        parent.columnconfigure(column, weight=1)
        
        label = ttk.Label(frame, text=label_text, style='Modern.TLabel')
        label.pack(anchor='w')
        
        slider = ttk.Scale(frame,
                          from_=from_,
                          to=to,
                          variable=variable,
                          orient=tk.HORIZONTAL,
                          style='Modern.Horizontal.TScale')
        slider.pack(fill=tk.X, pady=5)
        
        # Etichetta valore
        value_label = ttk.Label(frame, text=f"{variable.get():.2f}", style='Modern.TLabel')
        value_label.pack(anchor='e')
        
        def update_label(*args):
            value_label.config(text=f"{variable.get():.2f}")
        
        variable.trace_add('write', update_label)
        
    def load_image(self):
        """Carica un'immagine dal file system"""
        file_path = filedialog.askopenfilename(
            title="Seleziona un'immagine",
            filetypes=[
                ("Immagini", "*.jpg *.jpeg *.png *.bmp *.gif"),
                ("JPEG", "*.jpg *.jpeg"),
                ("PNG", "*.png"),
                ("Tutti i file", "*.*")
            ]
        )
        
        if file_path:
            try:
                self.original_image = Image.open(file_path)
                self.current_image = self.original_image.copy()
                
                # Reset di tutti i controlli e filtri
                self.brightness_var.set(1.0)
                self.contrast_var.set(1.0)
                self.temperature_var.set(0)
                
                # Reset stato filtri
                self.vintage_active = False
                self.bw_active = False
                self.update_filter_buttons()
                
                # Reset cronologia
                self.history = []
                self.save_to_history()
                
                self.display_current_image()
                self.canvas.delete("placeholder")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile caricare l'immagine:\n{str(e)}")
                
    def display_current_image(self):
        """Visualizza l'immagine corrente sul canvas"""
        if self.current_image is None:
            return
            
        # Calcola le dimensioni per il fit
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
            
        # Calcola il ridimensionamento mantenendo l'aspect ratio
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        new_width = int(img_width * ratio * 0.9)  # 90% per margini
        new_height = int(img_height * ratio * 0.9)
        
        # Ridimensiona l'immagine
        display_img = self.current_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Converti in PhotoImage
        self.display_image = ImageTk.PhotoImage(display_img)
        
        # Pulisci il canvas e mostra l'immagine
        self.canvas.delete("all")
        self.canvas.create_image(
            canvas_width // 2,
            canvas_height // 2,
            image=self.display_image,
            anchor=tk.CENTER,
            tags="image"
        )
        
    def save_to_history(self):
        """Salva lo stato corrente nella cronologia"""
        if self.original_image is None:
            return
            
        # Salva una copia dell'immagine e dello stato dei filtri
        state = {
            'image': self.original_image.copy(),
            'brightness': self.brightness_var.get(),
            'contrast': self.contrast_var.get(),
            'temperature': self.temperature_var.get(),
            'vintage': self.vintage_active,
            'bw': self.bw_active
        }
        
        self.history.append(state)
        
        # Limita la dimensione della cronologia
        if len(self.history) > self.max_history:
            self.history.pop(0)
            
    def undo(self):
        """Annulla l'ultima modifica"""
        if len(self.history) <= 1:
            messagebox.showinfo("Undo", "Nessuna operazione da annullare!")
            return
            
        # Rimuovi lo stato corrente e torna al precedente
        self.history.pop()
        
        # Ripristina lo stato precedente
        if self.history:
            previous_state = self.history[-1]
            self.original_image = previous_state['image'].copy()
            
            # Disattiva temporaneamente i trace per evitare salvataggi in cronologia
            self.brightness_var.trace_remove('write', self.brightness_trace_id)
            self.contrast_var.trace_remove('write', self.contrast_trace_id)
            self.temperature_var.trace_remove('write', self.temperature_trace_id)
            
            self.brightness_var.set(previous_state['brightness'])
            self.contrast_var.set(previous_state['contrast'])
            self.temperature_var.set(previous_state['temperature'])
            
            # Riattiva i trace
            self.brightness_trace_id = self.brightness_var.trace_add('write', self.apply_adjustments)
            self.contrast_trace_id = self.contrast_var.trace_add('write', self.apply_adjustments)
            self.temperature_trace_id = self.temperature_var.trace_add('write', self.apply_adjustments)
            
            self.vintage_active = previous_state['vintage']
            self.bw_active = previous_state['bw']
            
            self.update_filter_buttons()
            self.apply_adjustments()
            
    def update_filter_buttons(self):
        """Aggiorna lo stile dei pulsanti filtro in base allo stato"""
        if self.vintage_btn:
            style = 'Active.TButton' if self.vintage_active else 'Modern.TButton'
            self.vintage_btn.configure(style=style)
            
        if self.bw_btn:
            style = 'Active.TButton' if self.bw_active else 'Modern.TButton'
            self.bw_btn.configure(style=style)
    
    def create_placeholder(self):
        """Crea il testo placeholder centrato nel canvas"""
        if self.placeholder_text:
            self.canvas.delete(self.placeholder_text)
        
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        # Usa dimensioni di default se il canvas non è ancora renderizzato
        if canvas_width <= 1:
            canvas_width = 800
        if canvas_height <= 1:
            canvas_height = 600
        
        # Crea un placeholder più elegante con icona
        self.placeholder_text = self.canvas.create_text(
            canvas_width // 2,
            canvas_height // 2,
            text="📁 Carica un'immagine per iniziare",
            fill="#4a9eff",
            font=('Segoe UI', 18, 'bold'),
            tags="placeholder"
        )
        
    
    def on_canvas_resize(self, event):
        """Gestisce il ridimensionamento del canvas"""
        if self.original_image is None and self.placeholder_text:
            # Riposiziona il placeholder al centro
            self.create_placeholder()
        elif self.current_image is not None:
            # Ridisegna l'immagine per adattarla alle nuove dimensioni
            self.display_current_image()
    
    def get_base_image(self):
        """Ottiene l'immagine base senza filtri cromatici dalla cronologia"""
        # Cerca l'ultima immagine senza filtri vintage o b/n
        for i in range(len(self.history) - 1, -1, -1):
            state = self.history[i]
            if not state['vintage'] and not state['bw']:
                return state['image'].copy()
        
        # Se non trovata, usa la prima immagine della cronologia
        if self.history:
            return self.history[0]['image'].copy()
        
        # Fallback (non dovrebbe mai succedere)
        return self.original_image.copy()
        
    def apply_adjustments(self, *args):
        """Applica le regolazioni di luminosità, contrasto e temperatura"""
        if self.original_image is None:
            return
            
        # Parte dall'immagine originale
        img = self.original_image.copy()
        
        # Applica luminosità
        if self.brightness_var.get() != 1.0:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(self.brightness_var.get())
            
        # Applica contrasto
        if self.contrast_var.get() != 1.0:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(self.contrast_var.get())
            
        # Applica temperatura (modifica del bilanciamento del colore)
        if self.temperature_var.get() != 0:
            img = self.adjust_temperature(img, self.temperature_var.get())
            
        self.current_image = img
        self.display_current_image()
        
    def adjust_temperature(self, image, value):
        """Regola la temperatura del colore dell'immagine"""
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        # Converti in array numpy
        img_array = np.array(image, dtype=np.float32)
        
        # Applica shift di temperatura
        if value > 0:  # Più caldo (più rosso/giallo)
            img_array[:, :, 0] += value * 0.5  # Rosso
            img_array[:, :, 1] += value * 0.3  # Verde
        else:  # Più freddo (più blu)
            img_array[:, :, 2] += abs(value) * 0.5  # Blu
            
        # Clamp values
        img_array = np.clip(img_array, 0, 255)
        
        return Image.fromarray(img_array.astype(np.uint8))
        
    def toggle_vintage(self):
        """Attiva/disattiva il filtro vintage (seppia)"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Toggle dello stato
        self.vintage_active = not self.vintage_active
        
        # Se stiamo attivando vintage, disattiva B/N
        if self.vintage_active and self.bw_active:
            self.bw_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.vintage_active:
            # Applica filtro vintage sull'immagine base
            img = base_image.convert('RGB')
            img_array = np.array(img, dtype=np.float32)
            
            # Matrice seppia
            sepia_filter = np.array([
                [0.393, 0.769, 0.189],
                [0.349, 0.686, 0.168],
                [0.272, 0.534, 0.131]
            ])
            
            # Applica il filtro
            sepia_img = img_array @ sepia_filter.T
            sepia_img = np.clip(sepia_img, 0, 255)
            
            self.original_image = Image.fromarray(sepia_img.astype(np.uint8))
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
        
    def toggle_bw(self):
        """Attiva/disattiva il filtro bianco e nero"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Toggle dello stato
        self.bw_active = not self.bw_active
        
        # Se stiamo attivando B/N, disattiva Vintage
        if self.bw_active and self.vintage_active:
            self.vintage_active = False
        
        # Trova l'immagine base (senza filtri)
        base_image = self.get_base_image()
        
        if self.bw_active:
            # Applica filtro bianco e nero sull'immagine base
            self.original_image = base_image.convert('L').convert('RGB')
        else:
            # Rimuovi il filtro tornando all'immagine base
            self.original_image = base_image
        
        self.update_filter_buttons()
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
        
    def toggle_crop_mode(self):
        """Attiva/disattiva la modalità ritaglio"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.crop_mode = not self.crop_mode
        if self.crop_mode:
            self.canvas.config(cursor="cross")
            messagebox.showinfo("Modalità Ritaglio",
                              "Trascina il mouse sull'immagine per selezionare l'area da ritagliare.\n"
                              "Clicca di nuovo su 'Ritaglia' per applicare.")
        else:
            self.canvas.config(cursor="arrow")
            if self.crop_rect:
                self.apply_crop()
                
    def on_crop_start(self, event):
        """Inizia la selezione per il ritaglio"""
        if not self.crop_mode:
            return
        self.crop_start = (event.x, event.y)
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
    def on_crop_drag(self, event):
        """Disegna il rettangolo di selezione"""
        if not self.crop_mode or not self.crop_start:
            return
            
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
        self.crop_rect = self.canvas.create_rectangle(
            self.crop_start[0], self.crop_start[1],
            event.x, event.y,
            outline='#4a9eff',
            width=2,
            dash=(5, 5)
        )
        
    def on_crop_end(self, event):
        """Finalizza la selezione"""
        if not self.crop_mode or not self.crop_start:
            return
        self.crop_end = (event.x, event.y)
        
    def apply_crop(self):
        """Applica il ritaglio all'immagine"""
        if not self.crop_start or not self.crop_end:
            return
            
        # Calcola le coordinate relative all'immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Converti coordinate canvas in coordinate immagine
        x1 = int((min(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y1 = int((min(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        x2 = int((max(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y2 = int((max(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        
        # Clamp ai limiti dell'immagine
        x1 = max(0, min(x1, img_width))
        y1 = max(0, min(y1, img_height))
        x2 = max(0, min(x2, img_width))
        y2 = max(0, min(y2, img_height))
        
        if x2 > x1 and y2 > y1:
            self.original_image = self.current_image.crop((x1, y1, x2, y2))
            self.apply_adjustments()
            self.save_to_history()  # Salva DOPO la modifica
            
        self.crop_mode = False
        self.crop_start = None
        self.crop_end = None
        self.crop_rect = None
        self.canvas.config(cursor="arrow")
        
    def apply_mirror(self):
        """Applica l'effetto specchio (flip orizzontale)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.original_image = self.current_image.transpose(Image.FLIP_LEFT_RIGHT)
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
        
    def apply_rotation(self):
        """Ruota l'immagine di 90 gradi in senso orario"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.original_image = self.current_image.rotate(-90, expand=True)
        self.apply_adjustments()
        self.save_to_history()  # Salva DOPO la modifica
    
    def add_text(self):
        """Apre una finestra di dialogo per aggiungere testo all'immagine"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Crea finestra di dialogo con dimensioni maggiori
        dialog = tk.Toplevel(self.root)
        dialog.title("Aggiungi Testo")
        dialog.geometry("550x440")  # +50px larghezza, +40px altezza
        dialog.configure(bg="#2d2d2d")
        dialog.transient(self.root)
        dialog.grab_set()
        
        # Frame principale
        main_frame = ttk.Frame(dialog, style='Modern.TFrame', padding=20)
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Campo testo
        ttk.Label(main_frame, text="Testo:", style='Modern.TLabel', 
                 font=('Segoe UI', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        text_entry = tk.Text(main_frame, height=4, width=50, bg="#3d3d3d", fg="white", 
                            font=('Segoe UI', 11), insertbackground='white',
                            wrap=tk.WORD, relief=tk.FLAT, padx=10, pady=10)
        text_entry.pack(fill=tk.X, pady=(0, 20))
        text_entry.insert('1.0', 'Il tuo testo qui')
        text_entry.focus()
        text_entry.tag_configure("sel", background="#4a9eff")
        
        # Dimensione font
        ttk.Label(main_frame, text="Dimensione Font:", style='Modern.TLabel',
                 font=('Segoe UI', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        size_var = tk.IntVar(value=50)
        size_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        size_frame.pack(fill=tk.X, pady=(0, 20))
        size_slider = ttk.Scale(size_frame, from_=10, to=200, variable=size_var, 
                               orient=tk.HORIZONTAL, style='Modern.Horizontal.TScale')
        size_slider.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 10))
        size_label = ttk.Label(size_frame, text=f"{size_var.get()}", style='Modern.TLabel', 
                              width=5, font=('Segoe UI', 11, 'bold'))
        size_label.pack(side=tk.LEFT)
        
        def update_size_label(*args):
            size_label.config(text=f"{size_var.get()}")
        size_var.trace_add('write', update_size_label)
        
        # Colore
        ttk.Label(main_frame, text="Colore:", style='Modern.TLabel',
                 font=('Segoe UI', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        color_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        color_frame.pack(fill=tk.X, pady=(0, 20))
        
        color_var = tk.StringVar(value="#FFFFFF")
        selected_button = [None]  # Lista per tenere riferimento al bottone selezionato
        
        colors = [
            ("#FFFFFF", "Bianco"), ("#000000", "Nero"), ("#FF0000", "Rosso"),
            ("#00FF00", "Verde"), ("#0000FF", "Blu"), ("#FFFF00", "Giallo"),
            ("#FF00FF", "Magenta"), ("#00FFFF", "Ciano")
        ]
        
        color_buttons = []
        
        def select_color(hex_color, btn):
            color_var.set(hex_color)
            # Resetta tutti i bottoni
            for b in color_buttons:
                b.config(width=4, height=2, relief=tk.RAISED, borderwidth=2)
            # Ingrandisci il bottone selezionato
            btn.config(width=5, height=2, relief=tk.SUNKEN, borderwidth=4)
            selected_button[0] = btn
        
        for i, (hex_color, name) in enumerate(colors):
            # Crea un frame per ogni colore così posso centrarlo
            btn_container = tk.Frame(color_frame, bg="#2d2d2d")
            btn_container.pack(side=tk.LEFT, padx=3)
            
            btn = tk.Button(btn_container, bg=hex_color, width=4, height=2,
                          command=lambda c=hex_color, b=None: select_color(c, b),
                          relief=tk.RAISED, borderwidth=2, cursor="hand2")
            btn.pack()
            btn.config(command=lambda c=hex_color, b=btn: select_color(c, b))
            color_buttons.append(btn)
            
            # Aggiungi etichetta sotto il bottone per Mac
            lbl = tk.Label(btn_container, text=name.split()[0][:4], 
                          bg="#2d2d2d", fg="#999999", font=('Segoe UI', 8))
            lbl.pack()
            
            # Seleziona bianco di default
            if hex_color == "#FFFFFF":
                select_color(hex_color, btn)
        
        # Pulsanti azione
        button_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        button_frame.pack(fill=tk.X, pady=(15, 0))
        
        def on_ok():
            text_content = text_entry.get('1.0', 'end-1c').strip()
            if text_content:
                self.apply_text_to_image(text_content, size_var.get(), color_var.get())
                dialog.destroy()
            else:
                messagebox.showwarning("Attenzione", "Inserisci del testo!", parent=dialog)
        
        def on_cancel():
            dialog.destroy()
        
        ok_btn = ttk.Button(button_frame, text="Aggiungi", command=on_ok, 
                           style='Modern.TButton', width=15)
        ok_btn.pack(side=tk.LEFT, padx=5)
        
        cancel_btn = ttk.Button(button_frame, text="Annulla", command=on_cancel, 
                               style='Modern.TButton', width=15)
        cancel_btn.pack(side=tk.LEFT, padx=5)
        
        # Enter per confermare
        dialog.bind('<Return>', lambda e: on_ok())
        dialog.bind('<Escape>', lambda e: on_cancel())
    
    def apply_text_to_image(self, text, font_size, color):
        """Rende il testo mobile sul canvas per posizionamento personalizzato"""
        if self.current_image is None:
            return
        
        # Calcola il rapporto di scala tra canvas e immagine (IDENTICO a display_current_image)
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        
        # Applica 0.9 alle dimensioni, non al ratio (come in display_current_image)
        display_width = int(img_width * ratio * 0.9)
        display_height = int(img_height * ratio * 0.9)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Scala la dimensione del font per il canvas usando ratio * 0.9
        canvas_font_size = int(font_size * ratio * 0.9)
        
        # Posizione iniziale al centro dell'immagine visualizzata
        start_x = canvas_width // 2
        start_y = canvas_height // 2
        
        # Crea il testo sul canvas con outline VERO (usando più testi sovrapposti)
        outline_color = "black" if color == "#FFFFFF" else "white"
        
        # Crea gli outline sovrapposti per un effetto bordo vero
        outline_items = []
        offsets = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
        
        for dx, dy in offsets:
            outline = self.canvas.create_text(
                start_x + dx, start_y + dy,
                text=text,
                fill=outline_color,
                font=('Arial', canvas_font_size, 'bold'),
                tags="temp_text_outline"
            )
            outline_items.append(outline)
        
        # Crea il testo principale sopra gli outline
        canvas_text = self.canvas.create_text(
            start_x, start_y,
            text=text,
            fill=color,
            font=('Arial', canvas_font_size, 'bold'),
            tags="temp_text"
        )
        
        # Variabili per il drag
        drag_data = {
            "x": 0, 
            "y": 0, 
            "item": canvas_text, 
            "outlines": outline_items
        }
        
        def start_drag(event):
            drag_data["x"] = event.x
            drag_data["y"] = event.y
        
        def do_drag(event):
            dx = event.x - drag_data["x"]
            dy = event.y - drag_data["y"]
            self.canvas.move(drag_data["item"], dx, dy)
            for outline in drag_data["outlines"]:
                self.canvas.move(outline, dx, dy)
            drag_data["x"] = event.x
            drag_data["y"] = event.y
        
        def end_drag(event):
            pass  # Non serve salvare qui, lo faremo al momento dell'applicazione
        
        # Bind degli eventi per rendere il testo draggabile
        self.canvas.tag_bind("temp_text", "<ButtonPress-1>", start_drag)
        self.canvas.tag_bind("temp_text", "<B1-Motion>", do_drag)
        self.canvas.tag_bind("temp_text", "<ButtonRelease-1>", end_drag)
        
        # Crea dialog per conferma
        confirm_dialog = tk.Toplevel(self.root)
        confirm_dialog.title("Posiziona Testo")
        confirm_dialog.geometry("350x150")
        confirm_dialog.configure(bg="#2d2d2d")
        confirm_dialog.transient(self.root)
        
        frame = ttk.Frame(confirm_dialog, style='Modern.TFrame', padding=20)
        frame.pack(fill=tk.BOTH, expand=True)
        
        ttk.Label(frame, 
                 text="Trascina il testo nella posizione desiderata\npoi clicca Applica",
                 style='Modern.TLabel',
                 justify=tk.CENTER).pack(pady=20)
        
        button_frame = ttk.Frame(frame, style='Modern.TFrame')
        button_frame.pack()
        
        def apply_text():
            # Ottieni coordinate finali del testo sul canvas
            coords = self.canvas.coords(canvas_text)
            
            # Converti coordinate canvas in coordinate immagine usando ratio * 0.9
            scale_ratio = ratio * 0.9
            img_x = int((coords[0] - offset_x) / scale_ratio)
            img_y = int((coords[1] - offset_y) / scale_ratio)
            
            # Applica il testo all'immagine
            from PIL import ImageDraw, ImageFont
            
            img_with_text = self.original_image.copy()
            draw = ImageDraw.Draw(img_with_text)
            
            # Carica font con la dimensione ORIGINALE (non scalata)
            try:
                font_options = [
                    'arial.ttf', 'Arial.ttf',
                    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                    '/System/Library/Fonts/Helvetica.ttc',
                    'C:\\Windows\\Fonts\\arial.ttf'
                ]
                font = None
                for font_path in font_options:
                    try:
                        font = ImageFont.truetype(font_path, font_size)  # Usa font_size originale
                        break
                    except:
                        continue
                
                if font is None:
                    font = ImageFont.load_default()
            except:
                font = ImageFont.load_default()
            
            # Disegna bordo (outline) - bordo più sottile e preciso
            outline_col = "#000000" if color == "#FFFFFF" else "#FFFFFF"
            for adj_x in range(-1, 2):
                for adj_y in range(-1, 2):
                    if adj_x != 0 or adj_y != 0:
                        draw.text((img_x + adj_x, img_y + adj_y), text, font=font, fill=outline_col)
            
            # Disegna testo principale
            draw.text((img_x, img_y), text, font=font, fill=color)
            
            self.original_image = img_with_text
            self.apply_adjustments()
            self.save_to_history()  # Salva DOPO la modifica
            
            # Pulisci il canvas
            self.canvas.delete("temp_text")
            self.canvas.delete("temp_text_outline")
            confirm_dialog.destroy()
            
            messagebox.showinfo("Successo", "Testo applicato all'immagine!")
        
        def cancel_text():
            self.canvas.delete("temp_text")
            self.canvas.delete("temp_text_outline")
            confirm_dialog.destroy()
        
        apply_btn = ttk.Button(button_frame, text="Applica", command=apply_text, style='Modern.TButton')
        apply_btn.pack(side=tk.LEFT, padx=5)
        
        cancel_btn = ttk.Button(button_frame, text="Annulla", command=cancel_text, style='Modern.TButton')
        cancel_btn.pack(side=tk.LEFT, padx=5)
        
    def reset_image(self):
        """Resetta tutte le modifiche e torna all'immagine originale"""
        if not self.history:
            return
        
        # Torna alla primissima immagine caricata
        first_state = self.history[0]
        self.original_image = first_state['image'].copy()
        
        # Reset di tutti i controlli
        self.brightness_var.set(1.0)
        self.contrast_var.set(1.0)
        self.temperature_var.set(0)
        
        # Reset di tutti i filtri
        self.vintage_active = False
        self.bw_active = False
        self.update_filter_buttons()
        
        # Resetta la cronologia mantenendo solo lo stato originale
        self.history = [first_state]
        
        self.apply_adjustments()
        
    def save_image(self, format_type):
        """Salva l'immagine nel formato specificato"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Nessuna immagine da salvare!")
            return
            
        file_extension = f".{format_type}"
        file_types = [(format_type.upper(), f"*{file_extension}")]
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=file_extension,
            filetypes=file_types,
            title=f"Salva come {format_type.upper()}"
        )
        
        if file_path:
            try:
                if format_type.lower() == 'jpg':
                    # Converti in RGB se necessario (JPG non supporta trasparenza)
                    img_to_save = self.current_image
                    if img_to_save.mode in ('RGBA', 'LA', 'P'):
                        rgb_img = Image.new('RGB', img_to_save.size, (255, 255, 255))
                        if img_to_save.mode == 'P':
                            img_to_save = img_to_save.convert('RGBA')
                        rgb_img.paste(img_to_save, mask=img_to_save.split()[-1] if img_to_save.mode == 'RGBA' else None)
                        img_to_save = rgb_img
                    img_to_save.save(file_path, 'JPEG', quality=95)
                else:
                    self.current_image.save(file_path, 'PNG')
                    
                messagebox.showinfo("Successo", f"Immagine salvata con successo in:\n{file_path}")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile salvare l'immagine:\n{str(e)}")


def main():
    """Funzione principale"""
    root = tk.Tk()
    app = PhotoEditor(root)
    
    # Configura il ridimensionamento della finestra
    root.minsize(800, 600)
    
    # Centra la finestra
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()
